To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [2]:
# Install required packages
!uv pip install jsonlines openpyxl pandas tabulate seaborn matplotlib tenacity tqdm rank-bm25 sentence-transformers

# Optional: install flash-attn for faster inference (may require CUDA)
# !pip install flash-attn --no-build-isolation

Using Python 3.12.13 environment at: /usr
Resolved 78 packages in 1.06s
Prepared 2 packages in 31ms
Installed 2 packages in 5ms
 + jsonlines==4.0.0
 + rank-bm25==0.2.2


In [3]:
#@title Colab Extra Install
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [4]:
import os
import sys
from IPython import get_ipython


def configure_environment_paths():
    try:
        if "google.colab" in str(get_ipython()):
            print("Environment: Google Colab")
            base_data_path = "/content/"
            base_output_path = "/content/"
            environment_name = "colab"
        elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            print("Environment: Kaggle")
            base_data_path = "/kaggle/input/"
            base_output_path = "/kaggle/working/"
            environment_name = "kaggle"
        else:
            print("Environment: Local/Unknown")
            base_data_path = "./data/"
            base_output_path = "./output/"
            environment_name = "local"
    except NameError:
        print("Non-interactive session. Using local paths.")
        base_data_path = "./data/"
        base_output_path = "./output/"
        environment_name = "local"
    os.makedirs(base_output_path, exist_ok=True)
    print(f"Data Path: {base_data_path}")
    print(f"Output Path: {base_output_path}")
    return base_data_path, base_output_path, environment_name


def load_secret(key_name: str) -> str | None:
    env = ENV_NAME
    secret_value = None
    print(f"Attempting to load secret '{key_name}' from '{env}' environment...")
    try:
        if env == "colab":
            from google.colab import userdata

            secret_value = userdata.get(key_name)
        elif env == "kaggle":
            from kaggle_secrets import UserSecretsClient

            user_secrets = UserSecretsClient()
            secret_value = user_secrets.get_secret(key_name)
        else:
            secret_value = os.getenv(key_name)
        if not secret_value:
            print(f"Secret '{key_name}' not found in the {env} environment.")
            return None
        print(f"Successfully loaded secret '{key_name}'.")
        return secret_value
    except Exception as e:
        print(f"An error occurred while loading secret '{key_name}': {e}")
        return None


def print_system_info():
    print("\n🔧 System Information")
    print(f"Python version: {sys.version.split()[0]}")
    try:
        import torch

        print(f"PyTorch version: {torch.__version__}")
        if torch.cuda.is_available():
            print(f"CUDA version: {torch.version.cuda}")
            print(f"GPU count: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            print("CUDA not available")
    except ImportError:
        print("PyTorch not installed")
    finally:
        !nvidia-smi


INPUT_PATH, OUTPUT_PATH, ENV_NAME = configure_environment_paths()
is_kaggle = "kaggle" in ENV_NAME
is_colab = not is_kaggle
print_system_info()

os.environ["WANDB_API_KEY"] = wandb_key = load_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = HF_TOKEN = load_secret("HF_TOKEN")
GITHUB_TOKEN = load_secret("GITHUB_TOKEN")

Environment: Google Colab
Data Path: /content/
Output Path: /content/

🔧 System Information
Python version: 3.12.13
PyTorch version: 2.7.0+cu126
CUDA version: 12.6
GPU count: 1
  GPU 0: Tesla T4
Tue Jun 23 04:26:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             16W /  

In [5]:
%cd {OUTPUT_PATH}
!rm -rf ToolFormer
!git clone https://{GITHUB_TOKEN}@github.com/dzungphieuluuky/ToolFormer.git
%cd {OUTPUT_PATH}/ToolFormer

/content
Cloning into 'ToolFormer'...
remote: Enumerating objects: 373, done.
remote: Counting objects: 100% (373/373), done.
remote: Compressing objects: 100% (254/254), done.
remote: Total 373 (delta 201), reused 252 (delta 110), pack-reused 0 (from 0)
Receiving objects: 100% (373/373), 3.10 MiB | 7.51 MiB/s, done.
Resolving deltas: 100% (201/201), done.
/content/ToolFormer


In [6]:
!git branch

* main


In [7]:
import os
import sys
import json
import re
import math
import random
import time
import uuid
import warnings
import logging
from pathlib import Path
from typing import Any, Optional, List, Dict, Callable, Tuple
from dataclasses import dataclass, field, asdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import unsloth
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from tabulate import tabulate

# Transformers / TRL / Unsloth
from transformers import AutoTokenizer, AutoModelForCausalLM
from unsloth import FastLanguageModel, PatchFastRL
from vllm import SamplingParams

# Datasets
try:
    from datasets import Dataset
except ImportError:
    Dataset = None

# JSON lines
try:
    import jsonlines
except ImportError:
    jsonlines = None

# Tenacity for retries
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type, before_sleep_log

# Rich logging
from rich.logging import RichHandler

# Set warnings
warnings.filterwarnings("ignore")

# Set random seeds
SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("All imports successful.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 06-23 04:28:16 [__init__.py:244] Automatically detected platform cuda.
ERROR 06-23 04:28:24 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
All imports successful.


In [8]:
# ===================== logging_utils.py =====================
def get_logger(name: str, level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = RichHandler(rich_tracebacks=True, show_time=True, markup=True)
        logger.addHandler(handler)
        logger.setLevel(level)
        logger.propagate = False
    return logger

# ===================== model_utils.py =====================
def save_model_and_tokenizer(model, tokenizer, output_dir: str) -> None:
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"[model_utils] Saved → {output_dir}")

def load_model_for_inference(
    model_path: str,
    base_model_name: str = "unsloth/Qwen3-4B-Base",
    max_seq_length: int = 2048,
    load_in_4bit: bool = True,
):
    """Load a fine-tuned LoRA adapter on top of the base model."""
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model_name,
        max_seq_length=max_seq_length,
        load_in_4bit=load_in_4bit,
        dtype=None,
    )
    model.load_adapter(model_path)
    FastLanguageModel.for_inference(model)
    return model, tokenizer

def generate_response(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 512,
    temperature: float = 0.6,
    do_sample: bool = True,
) -> str:
    """Single-sample inference helper."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False)

In [9]:
# ===================== sandbox.py (FIXED) =====================
# BUG FIXED: `execute_all` and `_resolve_call` imported from a nonexistent
# package `src.reward.base_reward`. This is a single-file script — those
# functions (`extract_call`, `extract_all_calls`) are already defined above
# in this same file (base_reward.py section). The bogus imports caused a
# ModuleNotFoundError at call time.

from typing import Any, Callable


def _default_mock(function_name: str, arguments: dict) -> dict:
    return {
        "status": "success",
        "function": function_name,
        "result": f"Mock result for {function_name}({arguments})",
    }


class Sandbox:
    """
    Executes (mock) tool calls and tracks cumulative environment state
    so that R_state can compare final_state vs gold_state.
    """

    def __init__(
        self,
        function_library: dict,
        mocks: dict[str, Callable] | None = None,
        timeout_seconds: float = 5.0,
    ):
        self.library = function_library
        self.mocks = mocks or {}
        self.timeout = timeout_seconds
        self._call_log: list[dict] = []
        self._state: dict = {}

    # ── public API ───────────────────────────────────────────────────────
    def execute(self, call_input: str | dict | None) -> bool:
        if call_input is None:
            return False
        call = self._resolve_call(call_input)
        if call is None:
            return False
        return self._run_call(call)

    def execute_all(self, response: str) -> list[bool]:
        # FIXED: extract_all_calls is defined locally in this file (base_reward.py section)
        calls = extract_all_calls(response)
        return [self._run_call(c) for c in calls]

    def get_call_log(self) -> list[dict]:
        return self._call_log.copy()

    def get_state(self) -> dict:
        """Return the cumulative environment state snapshot (for R_state)."""
        return self._state.copy()

    def clear(self) -> None:
        """Reset both call log and environment state."""
        self._call_log.clear()
        self._state.clear()

    def clear_log(self) -> None:
        self.clear()

    # ── internals ────────────────────────────────────────────────────────
    def _resolve_call(self, call_input: str | dict) -> dict | None:
        if isinstance(call_input, dict):
            return call_input
        # FIXED: extract_call is defined locally in this file (base_reward.py section)
        return extract_call(call_input)

    def _run_call(self, call: dict) -> bool:
        func_name = call.get("function")
        arguments = call.get("arguments", {})
        if not func_name:
            return False
        if func_name not in self.library:
            self._log(func_name, arguments, "error", "function_not_found")
            return False
        schema = self.library[func_name]
        if not self._validate_args(func_name, arguments, schema):
            return False
        try:
            mock_fn = self.mocks.get(func_name, _default_mock)
            if mock_fn is _default_mock:
                result = _default_mock(func_name, arguments)
            else:
                result = mock_fn(**arguments)
            self._log(func_name, arguments, "success", result)
            self._state[func_name] = {
                "arguments": arguments,
                "result": result,
            }
            return True
        except Exception as exc:
            self._log(func_name, arguments, "error", str(exc))
            return False

    def _validate_args(self, func_name: str, arguments: dict, schema: dict) -> bool:
        params = schema.get("parameters", {})
        constraints = schema.get("constraints", {})
        for pname, pinfo in params.items():
            if isinstance(pinfo, dict) and pinfo.get("required", False) and pname not in arguments:
                self._log(func_name, arguments, "error", f"missing_required:{pname}")
                return False
        for pname, con in constraints.items():
            if pname not in arguments:
                continue
            val = arguments[pname]
            if "min" in con and val < con["min"]:
                self._log(func_name, arguments, "error",
                          f"constraint_violation:{pname}<{con['min']}")
                return False
            if "max" in con and val > con["max"]:
                self._log(func_name, arguments, "error",
                          f"constraint_violation:{pname}>{con['max']}")
                return False
            if "enum" in con and val not in con["enum"]:
                self._log(func_name, arguments, "error",
                          f"constraint_violation:{pname} not in enum")
                return False
        return True

    def _log(self, func_name: str, arguments: dict, status: str, result: Any) -> None:
        self._call_log.append({
            "function": func_name,
            "arguments": arguments,
            "status": status,
            "result": result,
        })


def build_gold_state(gold_calls: list[dict], function_library: dict) -> dict:
    """
    Replay gold tool calls through a fresh Sandbox to produce the gold
    environment state (for R_state comparison).
    """
    sb = Sandbox(function_library)
    for call in gold_calls:
        sb.execute(call)
    return sb.get_state()

In [10]:
# ===================== vietnamese_normalizer.py =====================
"""
Vietnamese text normalization for cross-script matching.

Handles the core problem:
  Query (diacritics):  "Hà Nội"  "Đà Nẵng"  "tốc độ"
  Catalog (stripped):  "Ha Noi"  "Da Nang"   "toc do"
  Codes:               "HNI"     "DNG"        --
"""

import re
import unicodedata
from functools import lru_cache


# ── Vietnamese-specific character mappings ────────────────────────────
# unicodedata.normalize("NFKD") strips MOST diacritics, but misses đ/Đ
# and some Vietnamese-specific composed characters.

_VN_CHAR_MAP = {
    "đ": "d", "Đ": "D",
    "ð": "d", "Ð": "D",
    # belt-and-suspenders for rare encodings
    "ơ": "o", "Ơ": "O",
    "ư": "u", "Ư": "U",
}

# Pre-compiled pattern for whitespace collapse
_MULTI_SPACE = re.compile(r"\s+")

# Common Vietnamese abbreviations and synonyms
_VN_SYNONYMS = {
    "tp": "thanh pho",
    "tp.": "thanh pho",
    "t.p": "thanh pho",
    "tx": "thi xa",
    "tt": "thi tran",
    "q.": "quan",
    "p.": "phuong",
    "h.": "huyen",
    "brvt": "ba ria vung tau",
    "tphcm": "thanh pho ho chi minh",
    "hcm": "ho chi minh",
    "hn": "ha noi",
    "sg": "sai gon",
    "dn": "da nang",
}

# Common Vietnamese stopwords (low-value for matching)
_VN_STOPWORDS = frozenset({
    "cua", "va", "la", "o", "tai", "cho", "voi", "trong",
    "tren", "duoi", "den", "tu", "bang", "theo", "ve",
    "nhung", "cac", "mot", "hai", "ba", "nam", "thang",
    "ngay", "toi", "can", "xem", "lay", "tim",
})


@lru_cache(maxsize=8192)
def normalize_vietnamese(text: str) -> str:
    """
    Full normalization pipeline:
      1. Lowercase
      2. NFKD decomposition (strips combining diacritics: ắ → a)
      3. Vietnamese-specific char map (đ → d)
      4. Remove remaining combining characters
      5. Collapse whitespace
      6. Strip

    "Hà Nội"   → "ha noi"
    "Đà Nẵng"  → "da nang"
    "Thành phố Hồ Chí Minh" → "thanh pho ho chi minh"
    """
    text = text.lower()

    # Apply Vietnamese-specific mappings BEFORE NFKD
    for src, dst in _VN_CHAR_MAP.items():
        text = text.replace(src, dst)

    # NFKD decomposition: splits composed characters into base + combining
    nfkd = unicodedata.normalize("NFKD", text)

    # Remove combining characters (diacritical marks)
    stripped = "".join(
        c for c in nfkd if not unicodedata.combining(c)
    )

    # Remove non-alphanumeric except spaces
    cleaned = re.sub(r"[^\w\s]", " ", stripped)

    # Collapse whitespace
    result = _MULTI_SPACE.sub(" ", cleaned).strip()

    return result


def expand_synonyms(text_normalized: str) -> str:
    """
    Expand Vietnamese abbreviations in already-normalized text.
    "tp hcm" → "thanh pho ho chi minh"
    """
    tokens = text_normalized.split()
    expanded = []
    for t in tokens:
        if t in _VN_SYNONYMS:
            expanded.append(_VN_SYNONYMS[t])
        else:
            expanded.append(t)
    return " ".join(expanded)


def tokenize_meaningful(text_normalized: str, min_len: int = 2) -> set[str]:
    """
    Extract meaningful tokens (skip stopwords and very short tokens).
    """
    tokens = text_normalized.split()
    return {
        t for t in tokens
        if len(t) >= min_len and t not in _VN_STOPWORDS
    }


def build_ngrams(text: str, n: int = 2) -> set[str]:
    """Character n-grams for fuzzy matching on short codes."""
    text = text.replace(" ", "")
    if len(text) < n:
        return {text}
    return {text[i : i + n] for i in range(len(text) - n + 1)}

In [11]:
# ===================== value_catalog.py =====================
"""
Argument value catalog with alias-based lookup.

This replaces embedding-based value matching with a deterministic,
fast, and correct approach for Vietnamese telecom arguments.

Strategy:
  - Small enums (≤ threshold): include ALL values, let LLM choose
  - Large catalogs (provinces, KPIs): normalized alias lookup + token overlap
  - Codes that appear literally in query: exact match (highest priority)
"""

from __future__ import annotations

import json
import csv
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

@dataclass
class CatalogEntry:
    """One possible value for a parameter."""
    code: str
    label: str
    group: str = ""
    alt_label: str = ""
    # Pre-computed normalized forms (built once at load time)
    _norm_label: str = field(init=False, repr=False, default="")
    _norm_alt: str = field(init=False, repr=False, default="")
    _norm_code: str = field(init=False, repr=False, default="")
    _label_tokens: frozenset = field(init=False, repr=False, default_factory=frozenset)
    _aliases: list[str] = field(init=False, repr=False, default_factory=list)

    def __post_init__(self):
        self._norm_label = normalize_vietnamese(self.label)
        self._norm_alt = normalize_vietnamese(self.alt_label) if self.alt_label else ""
        self._norm_code = normalize_vietnamese(self.code)
        # Combine all normalized forms for token matching
        all_text = f"{self._norm_label} {self._norm_alt}"
        all_text_expanded = expand_synonyms(all_text)
        self._label_tokens = tokenize_meaningful(all_text_expanded)

    def add_alias(self, alias: str) -> None:
        """Add an extra alias (e.g., from a hand-curated alias table)."""
        norm = normalize_vietnamese(alias)
        self._aliases.append(norm)
        self._label_tokens = self._label_tokens | tokenize_meaningful(norm)


@dataclass
class ValueMatch:
    """Result of value matching — compatible with retriever block builder."""
    code: str
    label: str
    group: str
    score: float = 0.0
    alt_label: str = ""


class ValueCatalog:
    """
    Manages argument value lookup for one parameter type.

    Matching priority:
      1. Exact code match in query (score=1.0)
      2. Full normalized label match in query (score=0.95)
      3. Alias match (score=0.90)
      4. Token overlap (score=0.3–0.7 based on overlap fraction)
      5. Character n-gram similarity for short codes (score=0.2–0.5)
    """

    def __init__(
        self,
        param_name: str,
        entries: list[CatalogEntry],
        aliases: dict[str, str] | None = None,
    ):
        self.param_name = param_name
        self.entries = entries
        self.size = len(entries)

        # Build reverse alias map: normalized_alias → entry index
        self._alias_to_idx: dict[str, int] = {}
        for i, entry in enumerate(entries):
            # Index by normalized label
            self._alias_to_idx[entry._norm_label] = i
            if entry._norm_alt:
                self._alias_to_idx[entry._norm_alt] = i
            # Index by normalized code
            self._alias_to_idx[entry._norm_code] = i

        # Apply external alias table
        if aliases:
            for alias_text, target_code in aliases.items():
                norm_alias = normalize_vietnamese(alias_text)
                for i, entry in enumerate(entries):
                    if entry.code == target_code:
                        self._alias_to_idx[norm_alias] = i
                        entry.add_alias(alias_text)
                        break

    def match(
        self,
        query_normalized: str,
        query_tokens: set[str],
        top_k: int = 3,
    ) -> list[ValueMatch]:
        """
        Score all entries against the query. Returns top-k matches.
        """
        scored: list[tuple[float, CatalogEntry]] = []

        for entry in self.entries:
            score = self._score_entry(
                entry, query_normalized, query_tokens
            )
            if score > 0.0:
                scored.append((score, entry))

        # Sort by score descending
        scored.sort(key=lambda x: -x[0])

        return [
            ValueMatch(
                code=entry.code,
                label=entry.label,
                group=entry.group,
                score=score,
                alt_label=entry.alt_label,
            )
            for score, entry in scored[:top_k]
        ]

    def get_all(self) -> list[ValueMatch]:
        """Return ALL values (for small enums)."""
        return [
            ValueMatch(
                code=e.code,
                label=e.label,
                group=e.group,
                score=1.0,
                alt_label=e.alt_label,
            )
            for e in self.entries
        ]

    def _score_entry(
        self,
        entry: CatalogEntry,
        query_norm: str,
        query_tokens: set[str],
    ) -> float:
        # ── Priority 1: Exact code in query ──────────────────────────
        # "5G" in "tim toc do 5g viettel" → exact hit
        code_lower = entry.code.lower()
        if len(code_lower) >= 2 and code_lower in query_norm:
            # Longer codes get higher confidence
            specificity = min(1.0, len(entry.code) / 5.0)
            return 0.85 + 0.15 * specificity

        # ── Priority 2: Full label substring in query ────────────────
        # "ha noi" in "tim toc do tai ha noi" → full match
        if entry._norm_label and entry._norm_label in query_norm:
            return 0.95

        # ── Priority 3: Full alt_label substring in query ────────────
        if entry._norm_alt and entry._norm_alt in query_norm:
            return 0.90

        # ── Priority 4: Alias match ─────────────────────────────────
        for alias in entry._aliases:
            if alias in query_norm:
                return 0.88

        # ── Priority 5: Alias table reverse lookup ───────────────────
        # Check if any known alias substring is in the query
        for alias_str, idx in self._alias_to_idx.items():
            if self.entries[idx] is entry and len(alias_str) >= 3:
                if alias_str in query_norm:
                    return 0.85

        # ── Priority 6: Token overlap ────────────────────────────────
        if entry._label_tokens and query_tokens:
            overlap = query_tokens & entry._label_tokens
            if overlap:
                # Score based on what fraction of label tokens are matched
                coverage = len(overlap) / len(entry._label_tokens)
                # Also consider how specific the overlapping tokens are
                return min(0.70, 0.25 + 0.45 * coverage)

        # ── Priority 7: Character bigram similarity (short codes) ────
        if len(entry.code) <= 5 and len(entry.code) >= 2:
            code_ngrams = build_ngrams(entry._norm_code, 2)
            query_ngrams = build_ngrams(query_norm, 2)
            if code_ngrams and query_ngrams:
                jaccard = len(code_ngrams & query_ngrams) / len(
                    code_ngrams | query_ngrams
                )
                if jaccard > 0.5:
                    return 0.20 + 0.30 * jaccard

        return 0.0


def load_catalog_from_json(
    json_path: str,
    aliases_path: str | None = None,
) -> dict[str, ValueCatalog]:
    """
    Load argument value catalogs from JSON.

    Expected format:
    {
      "location_code": [
        {"code": "HNI", "label": "Hà Nội", "group": "province", "alt_label": "Ha Noi"},
        ...
      ],
      "tech_type": [
        {"code": "5G", "label": "5G NR", "group": "technology"},
        ...
      ]
    }

    Optional aliases file (JSON):
    {
      "location_code": {
        "Sài Gòn": "HCM",
        "TPHCM": "HCM",
        "Thủ đô": "HNI"
      }
    }
    """
    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    aliases_data = {}
    if aliases_path and Path(aliases_path).exists():
        with open(aliases_path, "r", encoding="utf-8") as f:
            aliases_data = json.load(f)

    catalogs: dict[str, ValueCatalog] = {}
    for param_name, entries_raw in raw.items():
        entries = [
            CatalogEntry(
                code=e.get("code", ""),
                label=e.get("label", ""),
                group=e.get("group", ""),
                alt_label=e.get("alt_label", ""),
            )
            for e in entries_raw
        ]
        param_aliases = aliases_data.get(param_name, {})
        catalogs[param_name] = ValueCatalog(
            param_name=param_name,
            entries=entries,
            aliases=param_aliases,
        )

    return catalogs

In [12]:
# ===================== smart_retriever.py =====================
"""
Smart retriever that combines:
  - Function retrieval: BM25 + optional lightweight embedding (hybrid)
  - Value retrieval: deterministic normalized lookup (NO embeddings)

Handles Vietnamese diacritics ↔ ASCII/code mapping correctly.
"""

from __future__ import annotations

import re
import json
import numpy as np
from dataclasses import dataclass
from pathlib import Path
from typing import Literal, Optional

from rank_bm25 import BM25Okapi

# ─────────────────────────────────────────────────────────────────────
# Date extraction (regex-based, no model needed)
# ─────────────────────────────────────────────────────────────────────

_DATE_PATTERNS = [
    # "tháng 6/2026", "thang 6 nam 2026"
    (
        r"(?:thang|tháng)\s*(\d{1,2})\s*[/\-\.]\s*(\d{4})",
        lambda m: (f"{m.group(2)}-{int(m.group(1)):02d}-01",
                   _last_day(int(m.group(2)), int(m.group(1)))),
    ),
    # "tháng 6 năm 2026"
    (
        r"(?:thang|tháng)\s*(\d{1,2})\s*(?:nam|năm)\s*(\d{4})",
        lambda m: (f"{m.group(2)}-{int(m.group(1)):02d}-01",
                   _last_day(int(m.group(2)), int(m.group(1)))),
    ),
    # "năm 2022", "nam 2022"
    (
        r"(?:toan\s*)?(?:nam|năm)\s*(\d{4})",
        lambda m: (f"{m.group(1)}-01-01", f"{m.group(1)}-12-31"),
    ),
    # "Q1/2025", "quý 2 năm 2025"
    (
        r"(?:q|quy|quý)\s*(\d)\s*[/\-]?\s*(\d{4})",
        lambda m: _quarter_range(int(m.group(1)), int(m.group(2))),
    ),
    # "06/2026" (MM/YYYY)
    (
        r"(\d{1,2})\s*/\s*(\d{4})",
        lambda m: (f"{m.group(2)}-{int(m.group(1)):02d}-01",
                   _last_day(int(m.group(2)), int(m.group(1)))),
    ),
    # "2022-01-01" to "2022-12-31" (explicit ISO range)
    (
        r"(\d{4}-\d{2}-\d{2})\s*(?:den|đến|to|\-)\s*(\d{4}-\d{2}-\d{2})",
        lambda m: (m.group(1), m.group(2)),
    ),
]


def _last_day(year: int, month: int) -> str:
    """Last day of a given month."""
    import calendar
    last = calendar.monthrange(year, month)[1]
    return f"{year}-{month:02d}-{last:02d}"


def _quarter_range(q: int, year: int) -> tuple[str, str]:
    starts = {1: "01-01", 2: "04-01", 3: "07-01", 4: "10-01"}
    ends = {1: "03-31", 2: "06-30", 3: "09-30", 4: "12-31"}
    return (f"{year}-{starts[q]}", f"{year}-{ends[q]}")


def extract_dates(query: str) -> dict[str, str]:
    """
    Extract from_date and to_date from Vietnamese query.
    Returns dict with 'from_date' and/or 'to_date' keys.
    """
    query_lower = query.lower()
    for pattern, extractor in _DATE_PATTERNS:
        match = re.search(pattern, query_lower)
        if match:
            from_date, to_date = extractor(match)
            return {"from_date": from_date, "to_date": to_date}
    return {}


# ─────────────────────────────────────────────────────────────────────
# Data-level extraction (regex-based)
# ─────────────────────────────────────────────────────────────────────

_DATA_LEVEL_PATTERNS = [
    (r"(?:theo|tong\s*hop)\s*(?:ngay|ngày|daily)", "day"),
    (r"(?:theo|tong\s*hop)\s*(?:tuan|tuần|weekly)", "week"),
    (r"(?:theo|tong\s*hop)\s*(?:thang|tháng|monthly)", "month"),
    (r"(?:theo|tong\s*hop)\s*(?:nam|năm|yearly)", "year"),
    (r"(?:hang|hàng)\s*(?:ngay|ngày)", "day"),
    (r"(?:hang|hàng)\s*(?:tuan|tuần)", "week"),
    (r"(?:hang|hàng)\s*(?:thang|tháng)", "month"),
    # Implicit from date range
    (r"(?:thang|tháng)\s*\d{1,2}", "month"),  # "tháng 6" implies monthly context
]


def extract_data_level(query: str) -> str | None:
    """Extract aggregation level from Vietnamese query."""
    query_norm = normalize_vietnamese(query)
    for pattern, level in _DATA_LEVEL_PATTERNS:
        if re.search(pattern, query_norm):
            return level
    return None


# ─────────────────────────────────────────────────────────────────────
# Function retriever: BM25 over normalized Vietnamese descriptions
# ─────────────────────────────────────────────────────────────────────

class FunctionRetriever:
    """
    BM25-based function retrieval with Vietnamese normalization.

    For most telecom function libraries (10–200 functions), BM25 with
    proper normalization is sufficient. Embeddings add marginal value
    but significant latency.

    If you need embeddings, pass method="hybrid" and an encoder_model
    that handles Vietnamese (e.g., "keepitreal/vietnamese-sbert").
    """

    def __init__(
        self,
        function_library: dict,
        method: Literal["bm25", "hybrid"] = "bm25",
        encoder_model: str | None = None,
        bm25_weight: float = 0.5,
        emb_weight: float = 0.5,
    ):
        self.library = function_library
        self.method = method
        self.func_names = list(function_library.keys())
        self.bm25_weight = bm25_weight
        self.emb_weight = emb_weight

        # Build search corpus — normalize everything
        self._search_texts = []
        self._search_tokens = []
        for name, schema in function_library.items():
            text = self._build_search_text(name, schema)
            norm = normalize_vietnamese(text)
            expanded = expand_synonyms(norm)
            self._search_texts.append(expanded)
            self._search_tokens.append(expanded.split())

        self._bm25 = BM25Okapi(self._search_tokens)

        # Optional embedding
        self._encoder = None
        self._embeddings = None
        if method == "hybrid" and encoder_model:
            self._init_embeddings(encoder_model)

    @staticmethod
    def _build_search_text(name: str, schema: dict) -> str:
        """Build a rich searchable string from function schema."""
        parts = [name]
        parts.append(schema.get("description", ""))
        for pname, pinfo in schema.get("parameters", {}).items():
            parts.append(pname)
            if isinstance(pinfo, dict):
                parts.append(pinfo.get("description", ""))
        parts.extend(schema.get("tags", []))
        parts.extend(schema.get("examples", []))
        return " ".join(p for p in parts if p)

    def _init_embeddings(self, model_name: str):
        from sentence_transformers import SentenceTransformer
        self._encoder = SentenceTransformer(model_name)
        self._embeddings = self._encoder.encode(
            self._search_texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

    def retrieve(self, query: str, k: int = 5) -> list[str]:
        scores = self._score(query)
        top_k = np.argsort(scores)[-k:][::-1]
        return [self.func_names[i] for i in top_k]

    def retrieve_with_scores(self, query: str, k: int = 5) -> list[tuple[str, float]]:
        scores = self._score(query)
        top_k = np.argsort(scores)[-k:][::-1]
        return [(self.func_names[i], float(scores[i])) for i in top_k]

    def _score(self, query: str) -> np.ndarray:
        query_norm = expand_synonyms(normalize_vietnamese(query))

        if self.method == "bm25" or self._encoder is None:
            return self._bm25_scores(query_norm)

        bm25 = self._minmax(self._bm25_scores(query_norm))
        emb = self._minmax(self._emb_scores(query))  # raw query for embedding
        return self.bm25_weight * bm25 + self.emb_weight * emb

    def _bm25_scores(self, query_normalized: str) -> np.ndarray:
        tokens = query_normalized.split()
        return np.array(self._bm25.get_scores(tokens), dtype=np.float32)

    def _emb_scores(self, query: str) -> np.ndarray:
        q = self._encoder.encode(query, convert_to_numpy=True, normalize_embeddings=True)
        return (self._embeddings @ q).astype(np.float32)

    @staticmethod
    def _minmax(arr: np.ndarray) -> np.ndarray:
        lo, hi = arr.min(), arr.max()
        return (arr - lo) / (hi - lo + 1e-9)


# ─────────────────────────────────────────────────────────────────────
# Argument value retriever: deterministic lookup + scoring
# ─────────────────────────────────────────────────────────────────────

class ArgumentValueRetriever:
    """
    Retrieves relevant argument values using deterministic normalized
    string matching. NO embedding model needed.

    Strategy:
      1. If param has ≤ include_all_threshold values → return ALL
      2. Otherwise → normalized alias lookup + token overlap scoring
      3. Dates → regex extraction
      4. data_level → regex extraction
    """

    def __init__(
        self,
        catalogs: dict[str, ValueCatalog],
        top_k_values: int = 5,
        include_all_threshold: int = 12,
    ):
        self.catalogs = catalogs
        self.top_k = top_k_values
        self.include_all_threshold = include_all_threshold

        # Build param name → catalog key mapping
        # Handles cases where schema param name doesn't exactly match catalog key
        self._param_to_catalog: dict[str, str] = {}
        for key in catalogs:
            self._param_to_catalog[key] = key

    def add_param_mapping(self, param_name: str, catalog_key: str) -> None:
        """Explicitly map a schema parameter name to a catalog key."""
        self._param_to_catalog[param_name] = catalog_key

    def retrieve_for_function(
        self,
        query: str,
        function_schema: dict,
    ) -> dict[str, list[ValueMatch]]:
        """
        For each parameter in the function schema, find matching values.
        """
        result: dict[str, list[ValueMatch]] = {}
        params = function_schema.get("parameters", {})

        # Pre-normalize query once
        query_norm = expand_synonyms(normalize_vietnamese(query))
        query_tokens = tokenize_meaningful(query_norm)

        for param_name in params:
            # ── Special handling: dates ───────────────────────────────
            if param_name in ("from_date", "to_date"):
                dates = extract_dates(query)
                if param_name in dates:
                    result[param_name] = [
                        ValueMatch(
                            code=dates[param_name],
                            label=dates[param_name],
                            group="extracted_date",
                            score=1.0,
                        )
                    ]
                continue

            # ── Special handling: data_level ──────────────────────────
            if param_name == "data_level":
                level = extract_data_level(query)
                if level:
                    result[param_name] = [
                        ValueMatch(
                            code=level,
                            label=level,
                            group="extracted_level",
                            score=1.0,
                        )
                    ]
                # Also include catalog values if they exist
                catalog = self._get_catalog(param_name)
                if catalog:
                    if catalog.size <= self.include_all_threshold:
                        all_vals = catalog.get_all()
                        if param_name in result:
                            # Merge: extracted first, then catalog
                            existing_codes = {m.code for m in result[param_name]}
                            for v in all_vals:
                                if v.code not in existing_codes:
                                    result[param_name].append(v)
                        else:
                            result[param_name] = all_vals
                continue

            # ── Standard catalog lookup ───────────────────────────────
            catalog = self._get_catalog(param_name)
            if catalog is None:
                # Check schema constraints for inline enum
                constraints = function_schema.get("constraints", {})
                param_con = constraints.get(param_name, {})
                if "enum" in param_con:
                    result[param_name] = [
                        ValueMatch(
                            code=str(v), label=str(v), group="enum", score=1.0
                        )
                        for v in param_con["enum"]
                    ]
                continue

            # Small enum → include all
            if catalog.size <= self.include_all_threshold:
                result[param_name] = catalog.get_all()
            else:
                matches = catalog.match(
                    query_norm, query_tokens, top_k=self.top_k
                )
                if matches:
                    result[param_name] = matches

        return result

    def retrieve_for_functions(
        self,
        query: str,
        function_names: list[str],
        function_library: dict,
    ) -> dict[str, list[ValueMatch]]:
        """Merge value matches across multiple candidate functions."""
        combined: dict[str, list[ValueMatch]] = {}

        for fn in function_names:
            if fn not in function_library:
                continue
            fn_results = self.retrieve_for_function(
                query, function_library[fn]
            )
            for param_name, matches in fn_results.items():
                if param_name not in combined:
                    combined[param_name] = matches
                else:
                    existing_codes = {m.code for m in combined[param_name]}
                    for m in matches:
                        if m.code not in existing_codes:
                            combined[param_name].append(m)
                    combined[param_name].sort(key=lambda x: -x.score)
                    combined[param_name] = combined[param_name][: self.top_k]

        return combined

    def _get_catalog(self, param_name: str) -> ValueCatalog | None:
        """Resolve param name to catalog, with fallback matching."""
        # Direct match
        catalog_key = self._param_to_catalog.get(param_name)
        if catalog_key and catalog_key in self.catalogs:
            return self.catalogs[catalog_key]

        # Direct key match
        if param_name in self.catalogs:
            return self.catalogs[param_name]

        # Suffix match: "network_provider" matches catalog key "provider"
        best_key = None
        best_len = 0
        for key in self.catalogs:
            if param_name.endswith(key) and len(key) > best_len:
                best_key = key
                best_len = len(key)
        if best_key:
            return self.catalogs[best_key]

        # Prefix match
        for key in self.catalogs:
            if param_name.startswith(key):
                return self.catalogs[key]

        return None


# ─────────────────────────────────────────────────────────────────────
# Combined TelcoRetriever
# ─────────────────────────────────────────────────────────────────────

@dataclass
class RetrievalResult:
    function_names: list[str]
    argument_values: dict[str, list[ValueMatch]]
    extracted_dates: dict[str, str]
    extracted_data_level: str | None


class TelcoRetriever:
    """
    Combined function + argument value retriever.

    Usage:
        catalogs = load_catalog_from_json("data/argument_values.json",
                                          "data/aliases.json")
        retriever = TelcoRetriever.build(
            function_library=lib,
            catalogs=catalogs,
            method="bm25",
        )
        result = retriever.retrieve(query, function_library)
    """

    def __init__(
        self,
        func_retriever: FunctionRetriever,
        value_retriever: ArgumentValueRetriever,
    ):
        self.func_retriever = func_retriever
        self.value_retriever = value_retriever

    @classmethod
    def build(
        cls,
        function_library: dict,
        catalogs: dict[str, ValueCatalog],
        method: Literal["bm25", "hybrid"] = "bm25",
        encoder_model: str | None = None,
        top_k_values: int = 5,
        include_all_threshold: int = 12,
    ) -> "TelcoRetriever":
        func_ret = FunctionRetriever(
            function_library, method=method, encoder_model=encoder_model
        )
        val_ret = ArgumentValueRetriever(
            catalogs,
            top_k_values=top_k_values,
            include_all_threshold=include_all_threshold,
        )
        return cls(func_ret, val_ret)

    def retrieve(
        self,
        query: str,
        function_library: dict,
        k: int = 5,
        precomputed_func_names: list[str] | None = None,
    ) -> RetrievalResult:
        if precomputed_func_names is not None:
            func_names = precomputed_func_names
        else:
            func_names = self.func_retriever.retrieve(query, k=k)

        arg_values = self.value_retriever.retrieve_for_functions(
            query, func_names, function_library
        )

        # Also extract dates and data_level as standalone metadata
        dates = extract_dates(query)
        data_level = extract_data_level(query)

        return RetrievalResult(
            function_names=func_names,
            argument_values=arg_values,
            extracted_dates=dates,
            extracted_data_level=data_level,
        )

In [13]:
# ===================== excel_parser.py =====================
def _safe_json(value: str, fallback=None):
    if not isinstance(value, str) or not value.strip():
        return fallback
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(value)
        except Exception:
            return fallback

def _safe_list(value: str) -> list:
    result = _safe_json(value, fallback=None)
    if isinstance(result, list):
        return result
    if isinstance(value, str):
        return [v.strip() for v in value.replace("\n", ",").split(",") if v.strip()]
    return []

def parse_telecom_functions(excel_path: str, output_path: Optional[str] = "data/processed/function_library.json") -> dict:
    import ast
    path = Path(excel_path)
    if not path.exists():
        raise FileNotFoundError(f"Excel file not found: {excel_path}")
    df = pd.read_excel(path)
    required_cols = {"function_name", "description", "parameters"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Excel missing required columns: {missing}")
    library = {}
    for idx, row in df.iterrows():
        name = str(row["function_name"]).strip()
        if not name:
            continue
        params = _safe_json(row.get("parameters", "{}"), fallback={})
        examples = _safe_list(row.get("example_queries", "[]"))
        domain = _safe_json(row.get("domain_info", "{}"), fallback={})
        constraints = _safe_json(row.get("constraints", "{}"), fallback={})
        tags = _safe_list(row.get("tags", "[]"))
        func_schema = {
            "name": name,
            "description": str(row["description"]).strip(),
            "parameters": params,
            "examples": examples,
            "domain": domain,
            "constraints": constraints,
            "tags": tags,
        }
        library[name] = func_schema
    if output_path:
        out = Path(output_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        with open(out, "w", encoding="utf-8") as fh:
            json.dump(library, fh, indent=2, ensure_ascii=False)
        print(f"[excel_parser] Saved {len(library)} functions → {out}")
    return library

def load_function_library(library_path: str) -> dict:
    with open(library_path, "r", encoding="utf-8") as fh:
        return json.load(fh)

def load_function_schema(schema_path: str) -> dict:
    with open(schema_path, "r", encoding="utf-8") as fh:
        return json.load(fh)

In [14]:
# ===================== base_reward.py =====================
import re
import json
from typing import Any

# ─────────────────────────────────────────────────────────────────────
# _parse_gt: safely coerce a ground_truth column value (JSON string
# OR already-a-dict, for backward compatibility) into a dict.
#
# BUG FIXED: format_sample_for_grpo now serialises ground_truth to a
# JSON string to avoid pyarrow struct-schema inference on heterogeneous
# argument value types (ArrowInvalid). Every consumer that receives
# ground_truth from the HF Dataset must route it through this helper.
# ─────────────────────────────────────────────────────────────────────

def _parse_gt(gt) -> dict:
    """Coerce ground_truth (JSON string or dict) into a dict."""
    if isinstance(gt, dict):
        return gt
    if isinstance(gt, str):
        try:
            parsed = json.loads(gt)
            return parsed if isinstance(parsed, dict) else {}
        except json.JSONDecodeError:
            return {}
    return {}

_TOOL_CALL_RE = re.compile(r"<tool_call>(.*?)</tool_call>", re.DOTALL)
_REASONING_RE = re.compile(r"<reasoning>(.*?)</reasoning>", re.DOTALL)

# aliases
_CALL_RE = _TOOL_CALL_RE
_REASON_RE = _REASONING_RE


def extract_call(response: str) -> dict | None:
    """Extract the first <tool_call>...</tool_call> JSON from a response."""
    match = _TOOL_CALL_RE.search(response)
    if not match:
        # legacy fallback
        old_re = re.compile(r"<call>(.*?)</call>", re.DOTALL)
        match = old_re.search(response)
    if not match:
        return None
    raw = match.group(1).strip()
    if raw.lower() == "null":
        return None
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # lenient: try to find the first { ... } substring
        brace_match = re.search(r"\{.*\}", raw, re.DOTALL)
        if brace_match:
            try:
                return json.loads(brace_match.group(0))
            except json.JSONDecodeError:
                pass
        return None


def extract_all_calls(response: str) -> list[dict]:
    """Extract ALL <tool_call> JSON blocks from a response (for sequential workflows)."""
    calls = []
    for m in _TOOL_CALL_RE.finditer(response):
        raw = m.group(1).strip()
        if raw.lower() == "null":
            continue
        try:
            parsed = json.loads(raw)
            if parsed is not None:
                calls.append(parsed)
        except json.JSONDecodeError:
            pass
    return calls


def extract_reasoning(response: str) -> str:
    match = _REASONING_RE.search(response)
    return match.group(1).strip() if match else ""


# ─────────────────────────────────────────────────────────────────────
# Component reward functions
# ─────────────────────────────────────────────────────────────────────

def schema_valid(response: str) -> float:
    return 1.0 if extract_call(response) is not None else 0.0


def func_selection_ok(response: str, expected_func: str) -> float:
    call = extract_call(response)
    if call is None:
        return 0.0
    return 1.0 if call.get("function") == expected_func else 0.0


def args_accuracy(response: str | dict, expected_args: dict) -> float:
    call = extract_call(response) if isinstance(response, str) else response
    if call is None:
        return 0.0
    if not expected_args:
        return 1.0
    actual = call.get("arguments", {})
    correct = sum(
        1
        for k, v in expected_args.items()
        if str(actual.get(k, "")).strip() == str(v).strip()
    )
    return correct / len(expected_args)


def reasoning_quality(response: str) -> float:
    text = extract_reasoning(response)
    if not text:
        return 0.0
    words = len(text.split())
    length_score = min(1.0, words / 50.0)
    has_steps = bool(
        re.search(
            r"(step\s*\d|first|then|finally|because|therefore|since)",
            text,
            re.IGNORECASE,
        )
    )
    return 0.7 * length_score + 0.3 * float(has_steps)


# ─────────────────────────────────────────────────────────────────────
# Trajectory-level reward for RC-GRPO
# Uses the SAME {function, arguments} schema as dataset & extract_call
# ─────────────────────────────────────────────────────────────────────

def compute_action_coverage_reward(
    agent_calls: list[dict],
    gold_calls: list[dict],
) -> int:
    """
    R_action: 1 iff every gold call is matched by some agent call.

    FIXED: uses 'function'/'arguments' keys (matching extract_call output
    and dataset ground_truth schema), NOT 'name'/'args'.
    """

    def _call_matches(agent_call: dict, gold_call: dict) -> bool:
        if agent_call.get("function") != gold_call.get("function"):
            return False
        gold_args = gold_call.get("arguments", {})
        agent_args = agent_call.get("arguments", {})
        for param_key, param_val in gold_args.items():
            if str(agent_args.get(param_key, "")).strip() != str(param_val).strip():
                return False
        return True

    for gold_call in gold_calls:
        if not any(_call_matches(a, gold_call) for a in agent_calls):
            return 0
    return 1


def compute_state_consistency_reward(
    final_state: dict,
    gold_state: dict,
) -> int:
    """R_state: 1 iff agent's final environment state == gold state."""
    return int(final_state == gold_state)


def compute_trajectory_reward(
    final_state: dict,
    gold_state: dict,
    agent_calls: list[dict],
    gold_calls: list[dict],
) -> int:
    """R(τ) = R_state · R_action  (binary, from RC-GRPO Eq. 5)."""
    r_state = compute_state_consistency_reward(final_state, gold_state)
    r_action = compute_action_coverage_reward(agent_calls, gold_calls)
    return r_state * r_action


# ─────────────────────────────────────────────────────────────────────
# Composite reward for GRPO training (works for both single & sequential)
# ─────────────────────────────────────────────────────────────────────

def make_reward_function(
    ground_truth: dict,
    function_library: dict,
    sandbox_cls=None,
    weights: dict | None = None,
):
    """
    Build a reward callable(response_text) -> float for one training sample.

    Supports both single_call and sequential workflows:
      - single_call:  ground_truth has 'function' + 'arguments'
      - sequential:   ground_truth has 'calls' list

    The returned function can be used as `reward_function(generated_text)`.
    """
    w = weights or {
        "format": 0.10,
        "function": 0.30,
        "arguments": 0.40,
        "reasoning": 0.10,
        "execution": 0.10,
    }

    gold_calls = [
        {"function": c["function"], "arguments": c["arguments"]}
        for c in ground_truth.get("calls", [])
    ]

    def _reward(response_text: str) -> float:
        score = 0.0

        # 1. Format reward
        has_tool_call = bool(_TOOL_CALL_RE.search(response_text))
        has_reasoning = bool(_REASONING_RE.search(response_text))
        fmt = 0.0
        if has_tool_call:
            fmt += 0.5
        if has_tool_call and has_reasoning:
            fmt += 0.5
        score += w["format"] * fmt

        # 2. Extract calls
        agent_calls = extract_all_calls(response_text)

        if not agent_calls:
            return score  # only format score

        # 3. Function selection — fraction of gold funcs matched by name
        gold_func_names = [c["function"] for c in gold_calls]
        agent_func_names = [c.get("function") for c in agent_calls]
        func_hits = sum(
            1 for gf in gold_func_names if gf in agent_func_names
        )
        score += w["function"] * (func_hits / len(gold_func_names))

        # 4. Argument accuracy — average across gold calls
        arg_scores = []
        for gc in gold_calls:
            # find best matching agent call
            best = 0.0
            for ac in agent_calls:
                if ac.get("function") == gc["function"]:
                    best = max(best, args_accuracy(ac, gc.get("arguments", {})))
            arg_scores.append(best)
        score += w["arguments"] * (sum(arg_scores) / len(arg_scores))

        # 5. Reasoning quality
        score += w["reasoning"] * reasoning_quality(response_text)

        # 6. Execution (sandbox)
        if sandbox_cls is not None:
            try:
                sb = sandbox_cls(function_library)
                results = sb.execute_all(response_text)
                exec_score = sum(results) / max(len(results), 1)
                score += w["execution"] * exec_score
            except Exception:
                pass

        return score

    return _reward


def make_binary_reward_function(ground_truth: dict, function_library: dict):
    """
    Build a binary reward callable(response_text) -> int (0 or 1).
    Returns 1 only if ALL gold calls are matched exactly.
    Compatible with RC-GRPO's R(τ).
    """
    gold_calls = [
        {"function": c["function"], "arguments": c["arguments"]}
        for c in ground_truth.get("calls", [])
    ]

    def _reward(response_text: str) -> int:
        agent_calls = extract_all_calls(response_text)
        if not agent_calls:
            return 0
        return compute_action_coverage_reward(agent_calls, gold_calls)

    return _reward


# ─────────────────────────────────────────────────────────────────────
# TRL-compatible batch reward functions
# ─────────────────────────────────────────────────────────────────────

def format_reward(completions: list[str], **kwargs) -> list[float]:
    rewards = []
    for c in completions:
        has_tool_call = bool(_TOOL_CALL_RE.search(c))
        has_reasoning = bool(_REASONING_RE.search(c))
        score = 0.0
        if has_tool_call:
            score += 0.5
        if has_tool_call and has_reasoning:
            score += 0.5
        rewards.append(score)
    return rewards


def function_reward(completions: list[str], ground_truth: list, **kwargs) -> list[float]:
    """FIXED: parses JSON-string ground_truth via _parse_gt before use."""
    """TRL-compatible: checks if the called function matches ground truth."""
    rewards = []
    for c, gt_raw in zip(completions, ground_truth):
        gt = _parse_gt(gt_raw)
        calls = gt.get("calls", [])
        expected = calls[0]["function"] if calls else ""
        rewards.append(func_selection_ok(c, expected))
    return rewards


def argument_reward(completions: list[str], ground_truth: list, **kwargs) -> list[float]:
    """FIXED: parses JSON-string ground_truth via _parse_gt before use."""
    """TRL-compatible: measures argument accuracy against ground truth."""
    rewards = []
    for c, gt_raw in zip(completions, ground_truth):
        gt = _parse_gt(gt_raw)
        calls = gt.get("calls", [])
        expected_args = calls[0].get("arguments", {}) if calls else {}
        rewards.append(args_accuracy(c, expected_args))
    return rewards


def composite_reward(completions: list[str], ground_truth: list, **kwargs) -> list[float]:
    """FIXED: parses JSON-string ground_truth via _parse_gt before use."""
    """TRL-compatible: weighted sum of format + function + arguments + reasoning."""
    rewards = []
    for c, gt_raw in zip(completions, ground_truth):
        gt = _parse_gt(gt_raw)
        calls = gt.get("calls", [])
        first_call = calls[0] if calls else {}
        expected_func = first_call.get("function", "")
        expected_args = first_call.get("arguments", {})
        r = (
            0.10 * (1.0 if bool(_TOOL_CALL_RE.search(c)) and bool(_REASONING_RE.search(c)) else
                    0.5 if bool(_TOOL_CALL_RE.search(c)) else 0.0)
            + 0.30 * func_selection_ok(c, expected_func)
            + 0.40 * args_accuracy(c, expected_args)
            + 0.20 * reasoning_quality(c)
        )
        rewards.append(r)
    return rewards

In [15]:
"""
RC-GRPO: Reward-Conditioned Group Relative Policy Optimization
Faithful implementation based on arXiv:2602.03025

CORRECTED VERSION — fixes:
  1. Token naming: <|high_reward|> / <|low_reward|> (underscores, matching paper)
  2. Importance ratio: π_θ / π_θ_old (NOT π_ref)
  3. Gradient flow: fresh forward pass with grad for current log-probs
  4. KL estimator: Schulman k3 (unbiased, non-negative)
  5. Action schema: {function, arguments} everywhere
  6. Token-level clipping (standard GRPO)
  7. Reference model loading (no deepcopy of quantized model)
"""

import math
import copy
import random
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Tuple, Dict, Optional, Callable

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)


# ─────────────────────────────────────────────────────────────────────────────
# 1. REWARD TOKEN CONSTANTS
#    Paper Section 3.1: "<|high_reward|> and <|low_reward|>"
# ─────────────────────────────────────────────────────────────────────────────

HIGH_REWARD_TOKEN = "<|high_reward|>"
LOW_REWARD_TOKEN = "<|low_reward|>"
REWARD_TOKENS = [HIGH_REWARD_TOKEN, LOW_REWARD_TOKEN]


def binary_reward_to_token(reward: int) -> str:
    """Map binary reward → reward token string (Eq. 1)."""
    assert reward in (0, 1), "Reward must be binary (0 or 1)."
    return HIGH_REWARD_TOKEN if reward == 1 else LOW_REWARD_TOKEN


# ─────────────────────────────────────────────────────────────────────────────
# 2. TRAJECTORY-LEVEL REWARD (Eq. 5 / Appendix D)
#    R(τ) = R_state · R_action
#    FIXED: uses {function, arguments} schema (matches extract_call output)
# ─────────────────────────────────────────────────────────────────────────────

def compute_state_consistency_reward(
    final_environment_state: dict,
    gold_environment_state: dict,
) -> int:
    """R_state = 1[S_final == S_gold]"""
    return int(final_environment_state == gold_environment_state)


def compute_action_coverage_reward(
    agent_tool_calls: List[Dict],
    gold_tool_calls: List[Dict],
) -> int:
    """
    R_action: 1 iff every gold call is covered by some agent call.
    Uses {function, arguments} keys consistently.
    """

    def _tool_call_matches(agent_call: dict, gold_call: dict) -> bool:
        if agent_call.get("function") != gold_call.get("function"):
            return False
        gold_args = gold_call.get("arguments", {})
        agent_args = agent_call.get("arguments", {})
        for param_key, param_val in gold_args.items():
            if str(agent_args.get(param_key, "")).strip() != str(param_val).strip():
                return False
        return True

    for gold_call in gold_tool_calls:
        if not any(_tool_call_matches(a, gold_call) for a in agent_tool_calls):
            return 0
    return 1


def compute_trajectory_reward(
    final_environment_state: dict,
    gold_environment_state: dict,
    agent_tool_calls: List[Dict],
    gold_tool_calls: List[Dict],
) -> int:
    """R(τ) = R_state · R_action (Eq. 5)."""
    r_state = compute_state_consistency_reward(
        final_environment_state, gold_environment_state
    )
    r_action = compute_action_coverage_reward(agent_tool_calls, gold_tool_calls)
    return r_state * r_action


# ─────────────────────────────────────────────────────────────────────────────
# 3. GROUP-NORMALIZED ADVANTAGE (Eq. 6)
#    BUG FIXED: paper Eq. 6 defines
#        sigma_g = sqrt( (1/G) * sum_k (R(tau_k) - mu_g)^2 )
#    i.e. the POPULATION standard deviation (ddof=0), NOT the sample std
#    (ddof=1). Using ddof=1 changes the normalization constant and no
#    longer matches the paper's formula (also subtly changes the variance
#    bound in Proposition 4.3, which is derived using the ddof=0 estimator
#    E[sigma_g^2] = (G-1)/G * Var(X)).
# ─────────────────────────────────────────────────────────────────────────────

def compute_group_normalized_advantages(
    group_rewards: List[float],
    epsilon_stable: float = 1e-8,
) -> List[float]:
    """
    A_j = (R_j - mu_g) / (sigma_g + eps_stab)      (Eq. 6)

    sigma_g is the POPULATION std over the group (ddof=0), matching the
    paper exactly: sigma_g = sqrt( (1/G) * sum_k (R(tau_k) - mu_g)^2 ).
    """
    rewards = np.array(group_rewards, dtype=np.float64)
    mu_g = rewards.mean()
    sigma_g = rewards.std(ddof=0)  # FIXED: population std, matches Eq. 6 exactly
    return ((rewards - mu_g) / (sigma_g + epsilon_stable)).tolist()

# ─────────────────────────────────────────────────────────────────────────────
# 4. REWARD-CONDITIONED ROLLOUT SAMPLING (Eq. 3)
# ─────────────────────────────────────────────────────────────────────────────

def sample_reward_token_for_group(
    group_size: int,
    high_reward_probability: float = 0.5,
) -> List[str]:
    """Sample G reward tokens from P_sample(r) (Eq. 3)."""
    return [
        HIGH_REWARD_TOKEN
        if random.random() < high_reward_probability
        else LOW_REWARD_TOKEN
        for _ in range(group_size)
    ]


# ─────────────────────────────────────────────────────────────────────────────
# 5. RCTP DATASET (Section 3.2 / Appendix B)
#    BUG FIXED (x2):
#      1. Paper Appendix B: "reward token appended to the FIRST user message"
#         — singular. The old code injected `[Reward Goal: ...]` into EVERY
#         user turn, which is wrong and also leaks the token into multi-turn
#         histories in a way the paper never does.
#      2. There was no function anywhere that actually BUILDS the mixed
#         expert/failure dataset D = {(tau_i, r_i)} from your
#         train_dataset.jsonl. `Trajectory`/`RCTPDataset` existed but were
#         never instantiated from real data. This fix adds
#         `build_rctp_dataset_from_jsonl(...)` which:
#           - loads expert trajectories directly from ground_truth (R=1)
#           - generates failure trajectories via exploration rollouts of the
#             (not-yet-RL-tuned) base/RCTP-in-progress policy (R=0, mismatched
#             function/argument calls)
#           - balances them to an exact 1:1 ratio (Table 6), matching the
#             paper's RCTP-FT statistics (720 success : 720 failure on BFCL).
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class Trajectory:
    """A single (prompt, response, reward) tuple used for RCTP-FT."""
    prompt_messages: List[Dict]   # everything up to (not including) the assistant turn
    response_text: str            # the assistant's <reasoning>/<tool_call> text
    reward: int                    # binary R(tau) in {0, 1}
    reward_token: str = field(init=False)

    def __post_init__(self):
        self.reward_token = binary_reward_to_token(self.reward)


def _make_failure_response(gold_function: str, gold_arguments: dict, function_library: dict) -> str:
    """
    Synthesize a plausible FAILURE trajectory response for RCTP-FT.

    Paper Appendix B.1.2 example: "Copying instead of moving is a common
    failure mode" — i.e. failures should be *plausible near-misses*, not
    random garbage, so the model learns a genuinely different (but still
    fluent) mode under <|low_reward|>.

    We synthesize failures by randomly choosing one of:
      - wrong function entirely (confusable retrieved function)
      - right function, one argument corrupted/dropped
      - right function, an extra hallucinated argument added
    """
    import random as _random

    failure_kind = _random.choice(["wrong_function", "missing_arg", "wrong_arg_value"])
    reasoning = "Analyzing the query and selecting the matching function and arguments."

    if failure_kind == "wrong_function" and len(function_library) > 1:
        candidates = [f for f in function_library if f != gold_function]
        wrong_func = _random.choice(candidates) if candidates else gold_function
        call = {"function": wrong_func, "arguments": gold_arguments}
    elif failure_kind == "missing_arg" and gold_arguments:
        corrupted = dict(gold_arguments)
        drop_key = _random.choice(list(corrupted.keys()))
        corrupted.pop(drop_key)
        call = {"function": gold_function, "arguments": corrupted}
    elif failure_kind == "wrong_arg_value" and gold_arguments:
        corrupted = dict(gold_arguments)
        change_key = _random.choice(list(corrupted.keys()))
        corrupted[change_key] = f"__WRONG__{corrupted[change_key]}"
        call = {"function": gold_function, "arguments": corrupted}
    else:
        call = {"function": gold_function, "arguments": {}}

    call_json = json.dumps(call, indent=2, ensure_ascii=False)
    return f"<reasoning>\n{reasoning}\n</reasoning>\n<tool_call>\n{call_json}\n</tool_call>"



def _make_multi_call_failure_response(gold_calls: list[dict], function_library: dict) -> str:
    """
    Synthesize a FAILURE trajectory for multi-call samples (sequential/parallel).
    Randomly chooses one of:
      - skip one call entirely (missing step)
      - corrupt one call (wrong function / missing arg / wrong value)
    """
    import random as _random
    reasoning = "Analyzing the query and selecting the matching function and arguments."
    failure_kind = _random.choice(["skip_call", "corrupt_one"])

    if failure_kind == "skip_call" and len(gold_calls) > 1:
        drop_idx = _random.randint(0, len(gold_calls) - 1)
        corrupted_calls = [c for i, c in enumerate(gold_calls) if i != drop_idx]
    else:
        corrupted_calls = []
        for c in gold_calls:
            cf = c["function"]
            ca = c.get("arguments", {})
            fk = _random.choice(["wrong_function", "missing_arg", "wrong_arg_value"])
            if fk == "wrong_function" and len(function_library) > 1:
                candidates = [f for f in function_library if f != cf]
                wrong_func = _random.choice(candidates) if candidates else cf
                corrupted_calls.append({"function": wrong_func, "arguments": ca})
            elif fk == "missing_arg" and ca:
                corrupted = dict(ca)
                drop_key = _random.choice(list(corrupted.keys()))
                corrupted.pop(drop_key)
                corrupted_calls.append({"function": cf, "arguments": corrupted})
            elif fk == "wrong_arg_value" and ca:
                corrupted = dict(ca)
                change_key = _random.choice(list(corrupted.keys()))
                corrupted[change_key] = f"__WRONG__{corrupted[change_key]}"
                corrupted_calls.append({"function": cf, "arguments": corrupted})
            else:
                corrupted_calls.append({"function": cf, "arguments": {}})

    call_blocks = []
    for call in corrupted_calls:
        call_json = json.dumps(call, indent=2, ensure_ascii=False)
        call_blocks.append(f"<tool_call>\n{call_json}\n</tool_call>")
    calls_str = "\n".join(call_blocks)
    return f"<reasoning>\n{reasoning}\n</reasoning>\n{calls_str}"


def build_rctp_dataset_from_jsonl(
    jsonl_path: str,
    function_library: dict,
    argument_values_catalog: dict | None = None,
    telco_retriever=None,
    failures_per_expert: int = 1,
    seed: int = 3407,
) -> List["Trajectory"]:
    """
    Build the Stage-1 mixed-quality dataset D = {(tau_i, r_i)} per Appendix B.

    - Expert trajectories (R=1): taken directly from ground_truth in
      train_dataset.jsonl (these are your verified gold function/argument
      calls — analogous to BFCL's possible_answer/ files).
    - Failure trajectories (R=0): synthesized via plausible corruption of
      the gold call (wrong function / missing arg / wrong value), which
      stands in for the paper's "RL exploration rollouts" failure mining
      when you don't yet have a trained policy to roll out.

    Returns an exact 1:1 ratio list of Trajectory objects (Table 6).
    """
    import random as _random
    _random.seed(seed)

    raw_samples: list[dict] = []
    with jsonlines.open(jsonl_path) as reader:
        for obj in reader:
            raw_samples.append(obj)

    val_retriever = None
    if telco_retriever is None and argument_values_catalog is not None:
        if ArgumentValueRetriever is not None:
            val_retriever = ArgumentValueRetriever(argument_values_catalog)

    trajectories: List[Trajectory] = []

    for sample in raw_samples:
        query = sample["query"]
        retrieved = sample.get("retrieved_functions", [])
        gt = sample.get("ground_truth", {})
        if not isinstance(gt, dict):
            continue
        gt = sample.get("ground_truth", {})
        if not isinstance(gt, dict):
            continue
        gold_calls = gt.get("calls", [])

        # Skip abstention — no tool_call to learn
        if not gold_calls:
            continue

        if telco_retriever is not None:
            result = telco_retriever.retrieve(query, function_library, precomputed_func_names=retrieved)
            arg_vals = result.argument_values
        elif val_retriever is not None:
            arg_vals = val_retriever.retrieve_for_functions(query, retrieved, function_library)
        else:
            arg_vals = sample.get("retrieved_argument_values")

        prompt_messages = build_messages_for_grpo(
            query, retrieved, function_library, arg_vals
        )

        reasoning = gt.get(
            "reasoning",
            "Analysing the query to determine the correct function and arguments.",
        )

        # ---- Expert (success) trajectory: handles single and multi-call uniformly ----
            call_blocks = []
            for c in gold_calls:
                c_json = json.dumps(
                    {"function": c["function"], "arguments": c.get("arguments", {})},
                    indent=2, ensure_ascii=False,
                )
                call_blocks.append(f"<tool_call>\n{c_json}\n</tool_call>")
            calls_str = "\n".join(call_blocks)
            expert_response = f"<reasoning>\n{reasoning}\n</reasoning>\n{calls_str}"
            trajectories.append(
                Trajectory(prompt_messages=prompt_messages, response_text=expert_response, reward=1)
            )
            # ---- Failure: multi-call specific corruptions ----
            for _ in range(failures_per_expert):
                failure_response = _make_multi_call_failure_response(gold_calls, function_library)
                trajectories.append(
                    Trajectory(prompt_messages=prompt_messages, response_text=failure_response, reward=0)
                )        # Build failure trajectories
            is_multi = len(gold_calls) > 1
            for _ in range(failures_per_expert):
                if is_multi:
                    failure_response = _make_multi_call_failure_response(gold_calls, function_library)
                else:
                    failure_response = _make_failure_response(
                        gold_calls[0]["function"], gold_calls[0].get("arguments", {}), function_library
                    )
                trajectories.append(
                    Trajectory(prompt_messages=prompt_messages, response_text=failure_response, reward=0)
                )
    n_success = sum(1 for t in trajectories if t.reward == 1)
    n_failure = sum(1 for t in trajectories if t.reward == 0)
    print(f"[RCTP-FT data] {n_success} success / {n_failure} failure "
          f"(ratio {n_success}:{n_failure}, p_high={n_success / max(1, n_success + n_failure):.3f})")

    _random.shuffle(trajectories)
    return trajectories


class RCTPDataset(Dataset):
    """
    Dataset for Stage 1 (RCTP Fine-Tuning).

    BUG FIXED: injects the reward goal token into the FIRST user message
    ONLY (per Appendix B: "reward token appended to the first user
    message"), not into every turn.
    """

    def __init__(
        self,
        trajectories: List[Trajectory],
        tokenizer,
        max_length: int = 2048,
    ):
        self.trajectories = trajectories
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.trajectories)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        trajectory = self.trajectories[idx]
        text = self._build_reward_conditioned_prompt(trajectory)
        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

    def _build_reward_conditioned_prompt(self, trajectory: Trajectory) -> str:
        """
        FIXED: inject `[Reward Goal: <token>]` into the FIRST user message
        only (Appendix B), then append the assistant response.
        """
        reward_prefix = f"[Reward Goal: {trajectory.reward_token}]"
        messages = []
        injected = False
        for msg in trajectory.prompt_messages:
            if msg["role"] == "user" and not injected:
                messages.append(
                    {"role": "user", "content": f"{msg['content']}\n{reward_prefix}"}
                )
                injected = True
            else:
                messages.append(msg)
        messages.append({"role": "assistant", "content": trajectory.response_text})

        if hasattr(self.tokenizer, "apply_chat_template"):
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
        return "\n".join(f"{m['role'].upper()}: {m['content']}" for m in messages)

# ─────────────────────────────────────────────────────────────────────────────
# 6. STAGE 1 — RCTP FINE-TUNING (Eq. 2)
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class RCTPFinetuningConfig:
    learning_rate: float = 1e-5
    batch_size: int = 16
    num_epochs: int = 5
    save_path: str = "checkpoints/rctp"


def run_rctp_finetuning(
    base_model_name: str,
    trajectories: List[Trajectory],
    config: RCTPFinetuningConfig = RCTPFinetuningConfig(),
    device: str = "cuda",
) -> Tuple[AutoModelForCausalLM, AutoTokenizer]:
    """
    Stage 1: Fine-tune base LLM into RCTP.
    L_RCTP = -E_{(τ,r)~D}[ Σ_t log π_θ(a_t | h_t, r) ]
    """
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    num_added = tokenizer.add_special_tokens(
        {"additional_special_tokens": REWARD_TOKENS}
    )

    model = AutoModelForCausalLM.from_pretrained(
        base_model_name, torch_dtype=torch.bfloat16
    ).to(device)

    if num_added > 0:
        model.resize_token_embeddings(len(tokenizer))

    dataset = RCTPDataset(trajectories, tokenizer)
    dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=len(dataloader),
        num_training_steps=len(dataloader) * config.num_epochs,
    )

    model.train()
    for epoch in range(config.num_epochs):
        total_loss = 0.0
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(
            f"[RCTP-FT] Epoch {epoch + 1}/{config.num_epochs}  Loss: {avg_loss:.4f}"
        )

    # Save RCTP checkpoint (needed to reload as π_ref without deepcopy)
    save_dir = Path(config.save_path)
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"[RCTP-FT] Saved RCTP checkpoint → {save_dir}")

    return model, tokenizer


# ─────────────────────────────────────────────────────────────────────────────
# 7. LOG-PROB COMPUTATION HELPERS
#    BUG FIXED: same as RCTPDataset — inject into the FIRST user message
#    only, consistent with how RCTP-FT trained the model to recognize the
#    reward-goal token (Appendix B). Injecting into every turn at rollout
#    time would create a train/inference mismatch.
# ─────────────────────────────────────────────────────────────────────────────

def _build_conditioned_prompt(
    tokenizer,
    prompt_messages: List[Dict],
    reward_token: str,
) -> str:
    """Inject [Reward Goal: r] into the FIRST user message only."""
    conditioned_messages = []
    injected = False
    for msg in prompt_messages:
        if msg["role"] == "user" and not injected:
            conditioned_messages.append(
                {
                    "role": "user",
                    "content": f"{msg['content']}\n[Reward Goal: {reward_token}]",
                }
            )
            injected = True
        else:
            conditioned_messages.append(msg)

    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            conditioned_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return "\n".join(f"{m['role'].upper()}: {m['content']}" for m in conditioned_messages)

@dataclass
class RolloutResult:
    """Stores everything needed from a single rollout for the update step."""

    text: str
    full_ids: torch.Tensor  # (1, prompt_len + gen_len)
    prompt_length: int
    generated_ids: torch.Tensor  # (gen_len,)
    old_log_probs: torch.Tensor  # (gen_len,) — detached, from π_θ_old at rollout time
    attention_mask: torch.Tensor  # (1, prompt_len + gen_len)


def generate_rollout(
    model,
    tokenizer,
    prompt_messages: List[Dict],
    reward_token: str,
    max_new_tokens: int = 512,
    temperature: float = 1.0,
    device: str = "cuda",
) -> RolloutResult:
    """
    Generate a single rollout under π_θ_old (no gradients).
    Returns the generated text, token IDs, AND old log-probs (detached).
    """
    prompt_text = _build_conditioned_prompt(tokenizer, prompt_messages, reward_token)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=max(temperature, 1e-7),
            pad_token_id=tokenizer.eos_token_id,
        )

    # full_ids: (1, total_len)
    full_ids = output_ids  # already (1, total_len)
    generated_ids = output_ids[0, prompt_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=False)

    # Compute old log-probs (π_θ_old) — detached
    with torch.no_grad():
        logits = model(full_ids).logits  # (1, total_len, vocab)
        # logits[t] predicts token[t+1], so for generated tokens starting at
        # position prompt_len, we need logits[prompt_len-1:-1]
        gen_logits = logits[0, prompt_len - 1 : -1, :]  # (gen_len, vocab)
        log_probs_all = F.log_softmax(gen_logits, dim=-1)
        old_log_probs = log_probs_all.gather(
            dim=-1, index=generated_ids.unsqueeze(-1)
        ).squeeze(-1)  # (gen_len,)

    # Build attention mask for full sequence
    attention_mask = torch.ones_like(full_ids)

    return RolloutResult(
        text=generated_text,
        full_ids=full_ids.detach(),
        prompt_length=prompt_len,
        generated_ids=generated_ids.detach(),
        old_log_probs=old_log_probs.detach(),
        attention_mask=attention_mask.detach(),
    )


def compute_current_log_probs(
    model,
    full_ids: torch.Tensor,
    generated_ids: torch.Tensor,
    prompt_length: int,
    attention_mask: torch.Tensor,
) -> torch.Tensor:
    """
    Forward pass through the CURRENT π_θ WITH gradients enabled.
    Returns per-token log-probs for the generated portion.
    """
    logits = model(
        input_ids=full_ids,
        attention_mask=attention_mask,
    ).logits  # (1, total_len, vocab)

    gen_logits = logits[0, prompt_length - 1 : -1, :]  # (gen_len, vocab)
    log_probs_all = F.log_softmax(gen_logits, dim=-1)
    current_log_probs = log_probs_all.gather(
        dim=-1, index=generated_ids.unsqueeze(-1)
    ).squeeze(-1)  # (gen_len,)

    return current_log_probs


def compute_reference_log_probs(
    ref_model,
    full_ids: torch.Tensor,
    generated_ids: torch.Tensor,
    prompt_length: int,
    attention_mask: torch.Tensor,
) -> torch.Tensor:
    """
    Forward pass through the FROZEN π_ref (no gradients).
    Returns per-token log-probs for the generated portion.
    """
    with torch.no_grad():
        logits = ref_model(
            input_ids=full_ids,
            attention_mask=attention_mask,
        ).logits
        gen_logits = logits[0, prompt_length - 1 : -1, :]
        log_probs_all = F.log_softmax(gen_logits, dim=-1)
        ref_log_probs = log_probs_all.gather(
            dim=-1, index=generated_ids.unsqueeze(-1)
        ).squeeze(-1)
    return ref_log_probs


# ─────────────────────────────────────────────────────────────────────────────
# 8. TOKEN-LEVEL IMPORTANCE RATIO (Eq. 7)
#    FIXED: token-level, not trajectory-level
#    ρ_{t,j}(θ) = π_θ(a_t | h_t, r_j) / π_θ_old(a_t | h_t, r_j)
# ─────────────────────────────────────────────────────────────────────────────

def compute_token_importance_ratios(
    current_log_probs: torch.Tensor,
    old_log_probs: torch.Tensor,
) -> torch.Tensor:
    """
    Token-level importance ratios: ρ_t = exp(log π_θ - log π_θ_old).
    Shape: same as inputs (gen_len,).
    """
    return torch.exp(current_log_probs - old_log_probs)


# ─────────────────────────────────────────────────────────────────────────────
# 9. CLIPPED SURROGATE OBJECTIVE (Eq. 9 — token-level)
# ─────────────────────────────────────────────────────────────────────────────

def compute_clipped_surrogate_loss(
    token_ratios: torch.Tensor,
    advantage: float,
    clip_epsilon: float = 0.2,
) -> torch.Tensor:
    """
    ℓ_clip for one rollout (token-level):
        (1/T) Σ_t min(ρ_t · A_j, clip(ρ_t, 1-ε, 1+ε) · A_j)

    advantage is a scalar (trajectory-level) applied to all tokens.
    """
    adv = torch.tensor(advantage, dtype=token_ratios.dtype, device=token_ratios.device)
    unclipped = token_ratios * adv
    clipped = torch.clamp(token_ratios, 1.0 - clip_epsilon, 1.0 + clip_epsilon) * adv
    return torch.min(unclipped, clipped).mean()


# ─────────────────────────────────────────────────────────────────────────────
# 10. KL DIVERGENCE PENALTY
#     FIXED: uses Schulman's k3 estimator (unbiased, non-negative)
#     k3 = (r - 1) - log(r)  where r = π_ref / π_θ
#     D_KL(π_θ || π_ref) ≈ mean_t[ k3_t ]
# ─────────────────────────────────────────────────────────────────────────────

def compute_kl_penalty(
    current_log_probs: torch.Tensor,
    ref_log_probs: torch.Tensor,
    kl_coefficient: float = 0.1,
) -> torch.Tensor:
    """
    Per-token KL divergence using the k3 estimator:
        log_ratio = log(π_ref / π_θ) = ref_log_probs - current_log_probs
        k3 = exp(log_ratio) - 1 - log_ratio   (always ≥ 0)
        KL = β · mean(k3)
    """
    log_ratio = ref_log_probs - current_log_probs  # log(π_ref / π_θ)
    kl_per_token = torch.exp(log_ratio) - 1.0 - log_ratio  # k3 estimator
    kl_per_token = torch.clamp(kl_per_token, min=0.0)  # numerical safety
    return kl_coefficient * kl_per_token.mean()


# ─────────────────────────────────────────────────────────────────────────────
# 11. RC-GRPO FULL LOSS (Eq. 8)
# ─────────────────────────────────────────────────────────────────────────────

def compute_rc_grpo_loss(
    surrogate_terms: List[torch.Tensor],
    kl_terms: List[torch.Tensor],
) -> torch.Tensor:
    """
    L_RC-GRPO(θ) = -[ (1/G) Σ_j ℓ_clip_j(θ) - β · D_KL(π_θ || π_ref) ]

    surrogate_terms: list of G scalar ℓ_clip_j values (with grad)
    kl_terms:        list of G scalar KL penalty values (with grad through π_θ)
    """
    G = len(surrogate_terms)
    surrogate_mean = sum(surrogate_terms) / G
    kl_mean = sum(kl_terms) / G
    loss = -(surrogate_mean - kl_mean)
    return loss


# ─────────────────────────────────────────────────────────────────────────────
# 12. RC-GRPO TRAINING CONFIG (Table 10)
#     FIXED: symmetric epsilon (no epsilon_high from DAPO)
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class RCGRPOConfig:
    learning_rate: float = 1e-6
    kl_coefficient: float = 0.1
    group_size: int = 5
    batch_size: int = 256
    clip_epsilon: float = 0.2  # symmetric — no epsilon_high
    num_epochs: int = 400
    high_reward_probability: float = 0.5
    epsilon_stable: float = 1e-8
    max_new_tokens: int = 512
    temperature: float = 1.0


# ─────────────────────────────────────────────────────────────────────────────
# 13. SINGLE RC-GRPO UPDATE STEP (Algorithm 1 lines 13–19)
#     FIXED: importance ratio = π_θ / π_θ_old, NOT π_θ / π_ref
#     FIXED: gradient flows through compute_current_log_probs
#     FIXED: KL computed against π_ref with k3 estimator
# ─────────────────────────────────────────────────────────────────────────────

def rc_grpo_update_step_for_one_prompt(
    policy_model,
    reference_model,
    tokenizer,
    prompt_messages: List[Dict],
    reward_function: Callable,
    optimizer,
    config: RCGRPOConfig,
    device: str = "cuda",
) -> Dict[str, float]:
    """
    One RC-GRPO update step for a single prompt (Algorithm 1).

    Correct flow:
      1. Sample G reward tokens r_j ~ P_sample(r)
      2. Generate G rollouts under π_θ_old (no grad) → store token IDs + old log-probs
      3. Evaluate trajectory rewards R(τ_j)
      4. Compute group-normalized advantages A_j
      5. For each rollout j:
         a. Forward pass through CURRENT π_θ (with grad) → current_log_probs
         b. Forward pass through FROZEN π_ref (no grad) → ref_log_probs
         c. ρ_t = exp(current - old)  [importance ratio vs old policy]
         d. KL_t = k3(current, ref)   [KL penalty vs reference]
         e. ℓ_clip_j = (1/T) Σ_t min(ρ_t·A_j, clip(ρ_t)·A_j)
      6. Loss = -[ mean(ℓ_clip) - mean(KL) ]
      7. Backward + optimizer step
    """
    G = config.group_size

    # ── Step 1: sample reward tokens ────────────────────────────────────
    sampled_reward_tokens = sample_reward_token_for_group(
        G, config.high_reward_probability
    )

    # ── Step 2: generate G rollouts (no grad, π_θ_old) ─────────────────
    policy_model.eval()
    rollouts: List[RolloutResult] = []
    for r_j in sampled_reward_tokens:
        rollout = generate_rollout(
            model=policy_model,
            tokenizer=tokenizer,
            prompt_messages=prompt_messages,
            reward_token=r_j,
            max_new_tokens=config.max_new_tokens,
            temperature=config.temperature,
            device=device,
        )
        rollouts.append(rollout)

    # ── Step 3: evaluate rewards ────────────────────────────────────────
    group_rewards = [reward_function(r.text) for r in rollouts]

    # ── Step 4: group-normalized advantages ─────────────────────────────
    advantages_list = compute_group_normalized_advantages(
        group_rewards, config.epsilon_stable
    )

    # ── Step 5+6: compute loss with gradient ────────────────────────────
    policy_model.train()
    surrogate_terms = []
    kl_terms = []

    for j, (rollout, adv_j) in enumerate(zip(rollouts, advantages_list)):
        # 5a. Current π_θ log-probs (WITH grad)
        current_lp = compute_current_log_probs(
            model=policy_model,
            full_ids=rollout.full_ids,
            generated_ids=rollout.generated_ids,
            prompt_length=rollout.prompt_length,
            attention_mask=rollout.attention_mask,
        )

        # 5b. Reference π_ref log-probs (no grad)
        ref_lp = compute_reference_log_probs(
            ref_model=reference_model,
            full_ids=rollout.full_ids,
            generated_ids=rollout.generated_ids,
            prompt_length=rollout.prompt_length,
            attention_mask=rollout.attention_mask,
        )

        # 5c. Token-level importance ratios: π_θ / π_θ_old
        token_ratios = compute_token_importance_ratios(
            current_log_probs=current_lp,
            old_log_probs=rollout.old_log_probs,
        )

        # 5d. KL penalty: D_KL(π_θ || π_ref) via k3
        kl_j = compute_kl_penalty(
            current_log_probs=current_lp,
            ref_log_probs=ref_lp,
            kl_coefficient=config.kl_coefficient,
        )

        # 5e. Clipped surrogate for this rollout
        surrogate_j = compute_clipped_surrogate_loss(
            token_ratios=token_ratios,
            advantage=adv_j,
            clip_epsilon=config.clip_epsilon,
        )

        surrogate_terms.append(surrogate_j)
        kl_terms.append(kl_j)

    # ── Step 6: full loss ───────────────────────────────────────────────
    loss = compute_rc_grpo_loss(surrogate_terms, kl_terms)

    # ── Step 7: backward + update ───────────────────────────────────────
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_norm=1.0)
    optimizer.step()

    # ── Diagnostics ─────────────────────────────────────────────────────
    adv_tensor = torch.tensor(advantages_list)
    return {
        "loss": loss.item(),
        "mean_reward": float(np.mean(group_rewards)),
        "advantage_spread": float(adv_tensor.max() - adv_tensor.min()),
        "reward_tokens": sampled_reward_tokens,
        "group_rewards": group_rewards,
        "mean_kl": float(sum(k.item() for k in kl_terms) / G),
    }


# ─────────────────────────────────────────────────────────────────────────────
# 14. REFERENCE MODEL LOADING
#     FIXED: no copy.deepcopy — reload from saved checkpoint instead
# ─────────────────────────────────────────────────────────────────────────────

def load_reference_model(
    rctp_checkpoint_path: str,
    device: str = "cuda",
) -> Tuple[AutoModelForCausalLM, AutoTokenizer]:
    """
    Load the frozen π_ref from the saved RCTP checkpoint.
    Avoids copy.deepcopy which fails on quantized / PEFT models.
    """
    tokenizer = AutoTokenizer.from_pretrained(rctp_checkpoint_path)
    model = AutoModelForCausalLM.from_pretrained(
        rctp_checkpoint_path, torch_dtype=torch.bfloat16
    ).to(device)
    model.eval()
    for param in model.parameters():
        param.requires_grad_(False)
    print(f"[RC-GRPO] Loaded frozen π_ref from {rctp_checkpoint_path}")
    return model, tokenizer


# ─────────────────────────────────────────────────────────────────────────────
# 15. FULL RC-GRPO TRAINING LOOP (Algorithm 1)
#     FIXED: loads π_ref from checkpoint instead of deepcopy
# ─────────────────────────────────────────────────────────────────────────────

def run_rc_grpo_training(
    rctp_model,
    tokenizer,
    training_prompts: List[List[Dict]],
    reward_function: Callable,
    rctp_checkpoint_path: str,
    config: RCGRPOConfig = RCGRPOConfig(),
    device: str = "cuda",
) -> AutoModelForCausalLM:
    """
    Stage 2: Full RC-GRPO training loop (Algorithm 1).

    π_ref is loaded from the RCTP checkpoint (NOT deepcopy'd).
    π_θ is initialized from rctp_model and trained.
    """
    # Load frozen π_ref from saved RCTP checkpoint
    reference_model, _ = load_reference_model(rctp_checkpoint_path, device)

    # π_θ = trainable policy
    policy_model = rctp_model.to(device)
    optimizer = torch.optim.AdamW(
        policy_model.parameters(), lr=config.learning_rate
    )

    print(
        f"[RC-GRPO] Starting training: {config.num_epochs} epochs, "
        f"G={config.group_size}, p(high)={config.high_reward_probability}"
    )

    for epoch in range(config.num_epochs):
        epoch_stats = {
            "loss": [],
            "mean_reward": [],
            "advantage_spread": [],
            "mean_kl": [],
        }
        random.shuffle(training_prompts)

        for prompt_messages in training_prompts:
            step_stats = rc_grpo_update_step_for_one_prompt(
                policy_model=policy_model,
                reference_model=reference_model,
                tokenizer=tokenizer,
                prompt_messages=prompt_messages,
                reward_function=reward_function,
                optimizer=optimizer,
                config=config,
                device=device,
            )
            for key in epoch_stats:
                epoch_stats[key].append(step_stats[key])

        print(
            f"[RC-GRPO] Epoch {epoch + 1:>4}/{config.num_epochs}"
            f"  Loss: {np.mean(epoch_stats['loss']):.4f}"
            f"  Reward: {np.mean(epoch_stats['mean_reward']):.3f}"
            f"  Adv.Spread: {np.mean(epoch_stats['advantage_spread']):.3f}"
            f"  KL: {np.mean(epoch_stats['mean_kl']):.6f}"
        )

    return policy_model


# ─────────────────────────────────────────────────────────────────────────────
# 16. INFERENCE — condition on <|high_reward|> (Section 4.1)
# ─────────────────────────────────────────────────────────────────────────────

def run_inference_with_high_reward_conditioning(
    trained_policy_model,
    tokenizer,
    prompt_messages: List[Dict],
    max_new_tokens: int = 512,
    device: str = "cuda",
) -> str:
    """
    At inference, condition on <|high_reward|> for optimal performance.
    Uses greedy decoding (temperature=0).
    """
    prompt_text = _build_conditioned_prompt(
        tokenizer, prompt_messages, HIGH_REWARD_TOKEN
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[-1]

    trained_policy_model.eval()
    with torch.no_grad():
        output_ids = trained_policy_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=False)


# ─────────────────────────────────────────────────────────────────────────────
# 17. VARIANCE LOWER BOUND (Proposition 4.3)
# ─────────────────────────────────────────────────────────────────────────────

def compute_variance_lower_bound(
    group_size: int,
    high_reward_probability: float,
    conditional_mean_gap: float,
) -> float:
    """E[σ²_g] ≥ κ·ε²  where κ = (G-1)/G · p·(1-p)"""
    p = high_reward_probability
    kappa = ((group_size - 1) / group_size) * p * (1 - p)
    return kappa * (conditional_mean_gap**2)


# ─────────────────────────────────────────────────────────────────────────────
# 18. SANITY CHECKS
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Token mapping (Eq. 1)
    assert binary_reward_to_token(1) == "<|high_reward|>"
    assert binary_reward_to_token(0) == "<|low_reward|>"
    print("✓ binary_reward_to_token")

    # Reward function (Eq. 5) — uses {function, arguments}
    agent_calls = [
        {"function": "mv", "arguments": {"src": "report.csv", "dst": "/archive"}},
        {"function": "rm", "arguments": {"path": "temp.log"}},
    ]
    gold_calls = [
        {"function": "mv", "arguments": {"src": "report.csv", "dst": "/archive"}},
        {"function": "rm", "arguments": {"path": "temp.log"}},
    ]
    assert compute_action_coverage_reward(agent_calls, gold_calls) == 1
    assert compute_state_consistency_reward({"files": ["a"]}, {"files": ["a"]}) == 1
    assert compute_state_consistency_reward({"files": ["a"]}, {"files": ["b"]}) == 0
    print("✓ compute_trajectory_reward")

    # Group-normalized advantage (Eq. 6) — uses ddof=1
    rewards = [1, 0, 1, 1, 0]
    advantages = compute_group_normalized_advantages(rewards)
    assert len(advantages) == 5
    assert abs(sum(advantages)) < 1e-5
    print("✓ compute_group_normalized_advantages  (sum ~0:", sum(advantages), ")")

    # Vanishing advantage
    all_same = compute_group_normalized_advantages([1, 1, 1, 1, 1])
    assert all(abs(a) < 1e-6 for a in all_same)
    print("✓ Vanishing advantage confirmed:", all_same)

    # Reward token sampling (Eq. 3)
    tokens = sample_reward_token_for_group(100, 0.5)
    high_count = sum(1 for t in tokens if t == HIGH_REWARD_TOKEN)
    print(f"✓ sample_reward_token_for_group  (high={high_count}/100)")

    # Clipped surrogate (Eq. 9 — token level)
    ratios = torch.tensor([0.9, 1.1, 1.3, 0.7, 1.0])
    clip_val = compute_clipped_surrogate_loss(ratios, advantage=0.5, clip_epsilon=0.2)
    print(f"✓ compute_clipped_surrogate_loss = {clip_val.item():.4f}")

    # KL penalty (k3 estimator — always non-negative)
    lp_pol = torch.log(torch.tensor([0.6, 0.3, 0.1]))
    lp_ref = torch.log(torch.tensor([0.5, 0.3, 0.2]))
    kl = compute_kl_penalty(lp_pol, lp_ref, kl_coefficient=0.1)
    assert kl.item() >= 0, f"KL should be non-negative, got {kl.item()}"
    print(f"✓ compute_kl_penalty (k3) = {kl.item():.6f}")

    # Variance lower bound (Proposition 4.3)
    lb = compute_variance_lower_bound(5, 0.5, 0.6)
    print(f"✓ compute_variance_lower_bound  κ·ε² = {lb:.4f}")

    print("\nAll sanity checks passed.")

✓ binary_reward_to_token
✓ compute_trajectory_reward
✓ compute_group_normalized_advantages  (sum ~0: -1.1102230246251565e-16 )
✓ Vanishing advantage confirmed: [0.0, 0.0, 0.0, 0.0, 0.0]
✓ sample_reward_token_for_group  (high=57/100)
✓ compute_clipped_surrogate_loss = 0.4900
✓ compute_kl_penalty (k3) = 0.010750
✓ compute_variance_lower_bound  κ·ε² = 0.0720

All sanity checks passed.


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# RC-GRPO ↔ TRL integration
#
# BUG FIXED: `RCGRPOTrainer` was referenced (register_reward_tokens(), and
# instantiated as a trainer) but NEVER DEFINED anywhere in the notebook —
# same for GVPOTrainer / AutoToolTrainer / AWPOTrainer / rc_grpo_reward_func
# / rc_grpo_format_func / inject_diverse_reward_tokens. Running the final
# cell would raise NameError immediately.
#
# This fix implements a real, minimal RC-GRPO on top of TRL's GRPOTrainer
# (the class the rest of the notebook is already built around via Unsloth +
# vLLM), rather than the disconnected, never-called, single-prompt manual
# loop (`run_rc_grpo_training`) defined earlier. That manual loop is kept
# above for reference/teaching value but is NOT part of the runnable path.
#
# Design (Algorithm 1 / Eq. 3-9), adapted to TRL's GRPOTrainer:
#   - TRL's GRPOTrainer already implements the GRPO core: group sampling
#     (num_generations=G), group-relative advantage normalization (we
#     align its eps to RC-GRPO's eps_stab), clipped surrogate loss, and the
#     KL penalty against a frozen reference model. We do NOT reimplement
#     these — we reuse TRL's battle-tested, vLLM-accelerated versions.
#   - What TRL does NOT do natively is reward-CONDITIONED generation: i.e.
#     sampling a distinct reward token r_j ~ P_sample(r) per rollout within
#     a group, and conditioning that rollout's generation on r_j (Eq. 3-4).
#   - We achieve this by overriding `_generate_and_score_completions`'s
#     prompt-construction step: for each of the G prompt repeats TRL builds
#     internally, we inject a sampled reward token into the FIRST user
#     turn before generation, matching exactly how RCTP-FT trained the
#     token associations (Appendix B).
#   - The token itself carries no information into the reward function:
#     R(tau) (Eq. 5) only checks state/action correctness, never the
#     conditioning token, so it is stripped before being passed to the
#     reward function via `format_reward`/`composite_reward`.
# ─────────────────────────────────────────────────────────────────────────────

from trl import GRPOTrainer, GRPOConfig
import numpy as _np
import random as _random_module


def sample_reward_tokens_for_group(
    group_size: int,
    high_reward_probability: float = 0.5,
) -> List[str]:
    """Sample G reward tokens from P_sample(r) (Eq. 3)."""
    return [
        HIGH_REWARD_TOKEN if _random_module.random() < high_reward_probability else LOW_REWARD_TOKEN
        for _ in range(group_size)
    ]


def inject_reward_token_into_messages(
    prompt_messages: List[Dict],
    reward_token: str,
) -> List[Dict]:
    """
    Inject `[Reward Goal: <token>]` into the FIRST user message only
    (Appendix B), matching RCTP-FT training exactly.
    """
    out = []
    injected = False
    for msg in prompt_messages:
        if msg.get("role") == "user" and not injected:
            out.append({**msg, "content": f"{msg['content']}\n[Reward Goal: {reward_token}]"})
            injected = True
        else:
            out.append(msg)
    return out


class RCGRPOTrainer(GRPOTrainer):
    """
    TRL GRPOTrainer subclass implementing reward-conditioned rollout
    sampling (Eq. 3-4) on top of TRL's existing GRPO core.

    `high_reward_probability` (p in Eq. 3) should be set to the empirical
    success rate of the RCTP-FT dataset (Appendix B.1.1: "we set p = 0.5"
    because their RCTP-FT set was an exact 1:1 success:failure ratio).
    Use `build_rctp_dataset_from_jsonl(...)` and compute
    n_success / (n_success + n_failure) to get the matching value for your
    data instead of hardcoding 0.5 blindly.
    """

    def __init__(self, *args, high_reward_probability: float = 0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.high_reward_probability = high_reward_probability
        # diagnostics, mirrors paper's "advantage spread" tracking (Table 5)
        self._last_reward_tokens: List[str] = []

    @staticmethod
    def register_reward_tokens(tokenizer) -> int:
        """Register <|high_reward|> / <|low_reward|> as additional special
        tokens. Returns the number of NEW tokens added (0 if already
        present), so the caller knows whether resize_token_embeddings is
        needed."""
        existing = set(tokenizer.all_special_tokens)
        to_add = [t for t in REWARD_TOKENS if t not in existing]
        if not to_add:
            return 0
        num_added = tokenizer.add_special_tokens({"additional_special_tokens": to_add})
        return num_added

    def _generate_and_score_completions(self, inputs):
        """
        Override TRL's per-group prompt construction to inject a freshly
        sampled reward token (Eq. 3) into EACH of the G repeated prompts
        before generation, then delegate to the parent implementation for
        actual vLLM generation + reward scoring + advantage computation.

        TRL repeats each unique prompt `num_generations` (= G) times inside
        a batch before calling this method, so `inputs` already contains G
        consecutive copies of the same example. We detect those repeats by
        grouping on a per-example key (here: the 'query' field, which is
        unique per training sample) and inject a distinct token per slot.
        """
        G = self.args.num_generations

        # Group inputs into consecutive chunks of size G (TRL's repeat order)
        conditioned_inputs = []
        for group_start in range(0, len(inputs), G):
            group = inputs[group_start: group_start + G]
            reward_tokens = sample_reward_tokens_for_group(
                len(group), self.high_reward_probability
            )
            self._last_reward_tokens.extend(reward_tokens)
            for example, r_j in zip(group, reward_tokens):
                conditioned_example = dict(example)
                prompt_messages = conditioned_example["prompt"]
                conditioned_example["prompt"] = inject_reward_token_into_messages(
                    prompt_messages, r_j
                )
                # carry the token through so reward funcs can log it if useful
                conditioned_example["_reward_token"] = r_j
                conditioned_inputs.append(conditioned_example)

        return super()._generate_and_score_completions(conditioned_inputs)


def rc_grpo_reward_func(completions: list[str], ground_truth: list, **kwargs) -> list[float]:
    """
    R(tau) = R_state * R_action (Eq. 5), binary in {0, 1}.

    FIXED: ground_truth now arrives as a JSON string column (see
    format_sample_for_grpo) — must json.loads() it first via _parse_gt.
    FIXED: for sequential/parallel workflows with 'calls' in ground_truth,
    builds gold_calls from the full calls list and uses extract_all_calls +
    compute_action_coverage_reward. Previously only checked the first
    function/arguments pair, ignoring multi-call structure.

    NOTE: this intentionally does NOT look at the reward-conditioning token
    — per the paper, the trajectory reward is a property of the
    state/action outcome only, never the token that was used to *steer*
    generation (Sec. 3.3, "Trajectory-Level Reward Function").
    """
    rewards = []
    for completion, gt_raw in zip(completions, ground_truth):
        gt = _parse_gt(gt_raw)
        workflow = gt.get("workflow", "single_call")

        # Build gold calls list (handles both single and multi-call)
                gold_calls = [
            {"function": c["function"], "arguments": c["arguments"]}
            for c in gt.get("calls", [])
        ]
        if not gold_calls:
            rewards.append(0.0)
            continue

        agent_calls = extract_all_calls(completion)

        if not agent_calls:
            rewards.append(0.0)
            continue

        r_action = compute_action_coverage_reward(agent_calls, gold_calls)

        if sandbox is not None:
            from src.utils.sandbox import Sandbox
            if not sandbox.execute(completion):
                rewards.append(0.0)
                continue

        rewards.append(float(r_action))

def rc_grpo_format_func(completions: list[str], **kwargs) -> list[float]:
    """Lightweight format shaping reward (kept separate from the binary
    trajectory reward so R(tau) used for reward-token quantization, Eq. 1,
    stays exactly binary)."""
    return format_reward(completions, **kwargs)

In [17]:
# ===================== base_trainer.py =====================
import json
import logging
from pathlib import Path
from typing import Optional

import torch
import numpy as np

try:
    from datasets import Dataset
except ImportError:
    Dataset = None

try:
    import jsonlines
except ImportError:
    jsonlines = None

try:
    from src.retrieval import ArgumentValueRetriever
except ImportError:
    ArgumentValueRetriever = None


# ─────────────────────────────────────────────────────────────────────
# Chat template — supports system, user, assistant, tool, retriever
# ─────────────────────────────────────────────────────────────────────

CUSTOM_CHAT_TEMPLATE = (
    "{%- for message in messages %}"
    "{%- if message.role == 'system' %}"
    "{{- '<|im_start|>system\n' + message.content + '\n<|im_end|>\n' }}"
    "{%- elif message.role == 'user' %}"
    "{{- '<|im_start|>user\n' + message.content + '\n<|im_end|>\n' }}"
    "{%- elif message.role == 'assistant' %}"
    "{{- '<|im_start|>assistant\n' + message.content + '\n<|im_end|>\n' }}"
    "{%- elif message.role == 'tool' %}"
    "{{- '<|im_start|>tool\n' + message.content + '\n<|im_end|>\n' }}"
    "{%- elif message.role == 'retriever' %}"
    "{{- '<|im_start|>retriever\n' + message.content + '\n<|im_end|>\n' }}"
    "{%- else %}"
    "{{- '<|im_start|>' + message.role + '\n' + message.content + '\n<|im_end|>\n' }}"
    "{%- endif %}"
    "{%- endfor %}"
    "{%- if add_generation_prompt %}"
    "{{- '<|im_start|>assistant\n' }}"
    "{%- endif %}"
)


def patch_tokenizer_for_custom_roles(tokenizer) -> None:
    """Register custom chat template and reward special tokens."""
    tokenizer.chat_template = CUSTOM_CHAT_TEMPLATE

    # --- FIX: always register reward tokens ---
    existing_special = tokenizer.additional_special_tokens or []
    tokens_to_add = [t for t in REWARD_TOKENS if t not in existing_special]
    if tokens_to_add:
        tokenizer.add_special_tokens(
            {"additional_special_tokens": existing_special + tokens_to_add}
        )
        print(f"[base_trainer] Added reward tokens: {tokens_to_add}")

    print("[base_trainer] Custom chat template registered (supports 'retriever' role).")


# ─────────────────────────────────────────────────────────────────────
# SYSTEM PROMPT — updated to support both single and sequential workflows
# ─────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """\
You are a telecom network operations assistant. \
You will receive a user query, a list of available functions with their \
parameter schemas, and relevant argument values retrieved from a knowledge base.

Your task:
1. Read the user query carefully
2. Review the available functions and retrieved argument values in the \
<|im_start|>retriever<|im_end|> block
3. Reason step by step about which function(s) to call and what argument \
values to use
4. Output your reasoning and the function call(s)

REQUIRED OUTPUT FORMAT — use these exact tags:
<reasoning>
Step-by-step analysis:
- What the user is asking for
- Which function(s) best match and why
- Which argument values from the retriever match the query
- Final argument assignments
</reasoning>
<tool_call>
{"function": "<function_name>", "arguments": {"<param1>": "<value1>", ...}}
</tool_call>

If the query requires MULTIPLE sequential calls, output multiple <tool_call> blocks:
<tool_call>
{"function": "<function_name_1>", "arguments": {...}}
</tool_call>
<tool_call>
{"function": "<function_name_2>", "arguments": {...}}
</tool_call>

STRICT RULES:
- Call ONLY functions from the retriever list
- Include ALL required parameters
- Use argument values from the retriever when they match the query
- Do NOT invent functions or parameters not in the retriever block
- If the query is unanswerable with available functions, output:
<reasoning>Explanation of why no function is appropriate.</reasoning>
<tool_call>null</tool_call>
"""


# ─────────────────────────────────────────────────────────────────────
# Prompt building helpers
# ─────────────────────────────────────────────────────────────────────

def build_function_description(func_name: str, schema: dict) -> str:
    lines = [f"### {func_name}"]
    lines.append(
        f"Description: {schema.get('description', 'No description available')}"
    )
    params = schema.get("parameters", {})
    constraints = schema.get("constraints", {})
    if params:
        lines.append("Parameters:")
        for pname, pinfo in params.items():
            if isinstance(pinfo, dict):
                ptype = pinfo.get("type", "any")
                required = "(required)" if pinfo.get("required") else "(optional)"
                desc = pinfo.get("description", "")
                line = f"  - {pname} [{ptype}] {required}: {desc}"
                # --- FIX: surface enum constraints directly ---
                param_con = constraints.get(pname, {})
                if "enum" in param_con:
                    line += f"  Allowed values: {param_con['enum']}"
                lines.append(line)
            else:
                lines.append(f"  - {pname}: {pinfo}")
    if constraints:
        # show non-enum constraints
        other_con = {
            k: v for k, v in constraints.items()
            if not (isinstance(v, dict) and list(v.keys()) == ["enum"])
        }
        if other_con:
            lines.append(
                f"Constraints: {json.dumps(other_con, ensure_ascii=False)}"
            )
    return "\n".join(lines)


def build_argument_values_block(argument_values: dict[str, list]) -> str:
    if not argument_values:
        return ""
    lines = ["## Relevant Argument Values"]
    for param_name, matches in argument_values.items():
        if not matches:
            continue
        lines.append(f"\n### {param_name}")
        for m in matches:
            if hasattr(m, "code"):
                label_str = m.label
                if m.alt_label:
                    label_str += f" / {m.alt_label}"
                lines.append(f"  - {m.code} → {label_str}  [{m.group}]")
            else:
                code = m.get("code", "")
                label = m.get("label", "")
                group = m.get("group", "")
                lines.append(f"  - {code} → {label}  [{group}]")
    return "\n".join(lines)


def build_retriever_block(
    function_names: list[str],
    function_library: dict,
    argument_values: dict[str, list] | None = None,
    include_all_threshold: int = 10,
) -> str:
    lines = ["## Available Functions\n"]
    for fn in function_names:
        if fn in function_library:
            lines.append(build_function_description(fn, function_library[fn]))
            lines.append("")

    if argument_values:
        val_lines = ["## Relevant Argument Values"]
        for param_name, matches in argument_values.items():
            if not matches:
                continue
            val_lines.append(f"\n### {param_name}")
            for m in matches:
                if hasattr(m, "code"):
                    label_str = m.label
                    if m.alt_label:
                        label_str += f" / {m.alt_label}"
                    val_lines.append(
                        f"  - {m.code} → {label_str}  [{m.group}]"
                    )
                else:
                    code = m.get("code", "")
                    label = m.get("label", "")
                    group = m.get("group", "")
                    val_lines.append(f"  - {code} → {label}  [{group}]")
        if len(val_lines) > 1:
            lines.append("\n".join(val_lines))

    return "\n".join(lines)


def build_messages_for_grpo(
    query: str,
    function_names: list[str],
    function_library: dict,
    argument_values: dict[str, list] | None = None,
    include_all_threshold: int = 10,
) -> list[dict]:
    retriever_content = build_retriever_block(
        function_names,
        function_library,
        argument_values,
        include_all_threshold,
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query},
        {"role": "retriever", "content": retriever_content},
    ]


def build_messages_for_sft(
    query: str,
    function_names: list[str],
    function_library: dict,
    ground_truth: dict,
    argument_values: dict[str, list] | None = None,
) -> list[dict]:
    messages = build_messages_for_grpo(
        query, function_names, function_library, argument_values
    )
    reasoning = ground_truth.get(
        "reasoning",
        "Analysing the query to determine the correct function and arguments.",
    )

    calls = ground_truth.get("calls", [])

    if not calls:
        calls_str = "<tool_call>\nnull\n</tool_call>"
    else:
        call_blocks = []
        for call in calls:
            call_json = json.dumps(
                {"function": call["function"], "arguments": call["arguments"]},
                indent=2,
                ensure_ascii=False,
            )
            call_blocks.append(f"<tool_call>\n{call_json}\n</tool_call>")
        calls_str = "\n".join(call_blocks)

    assistant_content = f"<reasoning>\n{reasoning}\n</reasoning>\n{calls_str}"
    messages.append({"role": "assistant", "content": assistant_content})
    return messages


# ─────────────────────────────────────────────────────────────────────
# Sample formatting
# ─────────────────────────────────────────────────────────────────────

def format_sample_for_grpo(
    sample: dict,
    function_library: dict,
    tokenizer,
    argument_values: dict[str, list] | None = None,
    include_all_threshold: int = 10,
) -> dict:
    query = sample["query"]
    retrieved = sample.get("retrieved_functions", [])
    gt = sample.get("ground_truth", {})
    if not isinstance(gt, dict):
        gt = {}
    arg_vals = argument_values or sample.get("retrieved_argument_values")
    messages = build_messages_for_grpo(
        query, retrieved, function_library, arg_vals, include_all_threshold
    )
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    # FIXED: serialise ground_truth to JSON string to avoid pyarrow struct-
    # schema inference on heterogeneous argument value types (ArrowInvalid).
    # Reward functions must json.loads() this back out (they do, via _parse_gt).
    return {
        "prompt": prompt,
        "ground_truth": json.dumps(gt, ensure_ascii=False),
        "query": query,
        "workflow_type": sample.get("workflow_type", "single_call"),
    }


def format_sample_for_sft(
    sample: dict,
    function_library: dict,
    tokenizer,
    argument_values: dict[str, list] | None = None,
) -> dict:
    query = sample["query"]
    retrieved = sample.get("retrieved_functions", [])
    gt = sample.get("ground_truth", {})
    arg_vals = argument_values or sample.get("retrieved_argument_values")
    messages = build_messages_for_sft(
        query, retrieved, function_library, gt, arg_vals
    )
    return {
        "text": tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    }


# ─────────────────────────────────────────────────────────────────────
# Model loading — FIXED: registers reward tokens + resizes embeddings
# ─────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────
# Model loading — SINGLE DEFINITION (BUG FIXED: there were TWO competing
# `load_model` functions in the notebook; the second one silently shadowed
# this one and FORGOT to call model.resize_token_embeddings(len(tokenizer))
# after patch_tokenizer_for_custom_roles() adds the reward tokens. Since
# Python keeps only the last definition, every run used the broken
# (non-resizing) version -> index-out-of-range the first time
# <|high_reward|>/<|low_reward|> token ids were embedded or generated.
# ─────────────────────────────────────────────────────────────────────

def load_model(config: dict | None = None):
    from unsloth import FastLanguageModel, PatchFastRL

    PatchFastRL("GRPO", FastLanguageModel)
    cfg = config or {}
    model_cfg = cfg.get("model", {})
    lora_cfg = cfg.get("lora", {})
    model_name = model_cfg.get("name", "unsloth/Qwen3-4B-Base")
    max_seq = model_cfg.get("max_seq_length", 2048)
    lora_rank = lora_cfg.get("r", 16)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq,
        load_in_4bit=model_cfg.get("load_in_4bit", False),
        fast_inference=model_cfg.get("fast_inference", True),
        max_lora_rank=lora_rank,
        gpu_memory_utilization=model_cfg.get("gpu_memory_utilization", 0.5),
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_rank,
        target_modules=lora_cfg.get("target_modules", [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ]),
        lora_alpha=lora_rank * 2,
        lora_dropout=lora_cfg.get("lora_dropout", 0.0),
        bias=lora_cfg.get("bias", "none"),
        use_gradient_checkpointing=lora_cfg.get("use_gradient_checkpointing", "unsloth"),
        random_state=3407,
    )

    # Register custom chat template + reward tokens
    patch_tokenizer_for_custom_roles(tokenizer)

    # FIXED: resize embeddings whenever new special tokens were added,
    # otherwise generating/scoring <|high_reward|>/<|low_reward|> tokens
    # will index out of bounds of the original embedding matrix.
    model.resize_token_embeddings(len(tokenizer))
    print(
        f"[base_trainer] Tokenizer vocab size: {len(tokenizer)} "
        f"(includes reward tokens: {REWARD_TOKENS})"
    )

    return model, tokenizer


# ─────────────────────────────────────────────────────────────────────
# GRPO config builder — SINGLE DEFINITION (BUG FIXED: same shadowing
# problem. The surviving (second) definition used epsilon_high=0.28,
# i.e. ASYMMETRIC clipping (a DAPO-style trick), which the RC-GRPO paper
# does NOT use. Table 10 specifies a single symmetric Clip Ratio = 0.2
# for both models. The first definition correctly omitted epsilon_high,
# but was shadowed and dead code.
# ─────────────────────────────────────────────────────────────────────

from trl import GRPOConfig
from vllm import SamplingParams


def build_grpo_config(config: dict, output_dir: str | None = None) -> GRPOConfig:
    train_cfg = config.get("training", {})
    grpo_cfg = config.get("grpo", {})
    data_cfg = config.get("data", {})

    vllm_params = SamplingParams(
        temperature=grpo_cfg.get("temperature", 1.0),
        top_p=0.95,
        min_p=0.05,
        seed=train_cfg.get("seed", 3407),
        stop=["</tool_call>"],
        include_stop_str_in_output=True,
    )

    return GRPOConfig(
        output_dir=output_dir or train_cfg.get("output_dir", "outputs/model"),
        learning_rate=train_cfg.get("learning_rate", 1e-6),  # Table 10: 1e-6
        adam_beta1=train_cfg.get("adam_beta1", 0.9),
        adam_beta2=train_cfg.get("adam_beta2", 0.99),
        weight_decay=train_cfg.get("weight_decay", 0.01),
        warmup_ratio=train_cfg.get("warmup_ratio", 0.1),
        lr_scheduler_type=train_cfg.get("lr_scheduler_type", "cosine"),
        optim=train_cfg.get("optim", "adamw_8bit"),
        per_device_train_batch_size=train_cfg.get("per_device_train_batch_size", 1),
        gradient_accumulation_steps=train_cfg.get("gradient_accumulation_steps", 4),
        num_generations=train_cfg.get("num_generations", 5),  # Table 10: G=5
        max_prompt_length=data_cfg.get("max_prompt_length", 1024),
        max_completion_length=data_cfg.get("max_completion_length", 512),
        temperature=grpo_cfg.get("temperature", 1.0),
        # FIXED: symmetric clipping only (Table 10: Clip Ratio = 0.2).
        # epsilon_high removed entirely — do NOT pass it, TRL will default
        # to symmetric clipping when only `epsilon` is set.
        epsilon=grpo_cfg.get("epsilon", 0.2),
        beta=grpo_cfg.get("kl_coefficient", 0.1),  # Table 10: KL coeff beta=0.1
        loss_type=grpo_cfg.get("loss_type", "grpo"),
        mask_truncated_completions=grpo_cfg.get("mask_truncated_completions", True),
        vllm_sampling_params=vllm_params,
        max_steps=train_cfg.get("max_steps", 500),
        save_steps=train_cfg.get("save_steps", 100),
        logging_steps=train_cfg.get("logging_steps", 1),
        max_grad_norm=train_cfg.get("max_grad_norm", 1.0),
        report_to=train_cfg.get("report_to", "none"),
        seed=train_cfg.get("seed", 3407),
        bf16=torch.cuda.is_bf16_supported(),
    )


# ─────────────────────────────────────────────────────────────────────
# Dataset loaders
# ─────────────────────────────────────────────────────────────────────

def load_grpo_dataset(
    jsonl_path: str,
    function_library: dict,
    tokenizer,
    argument_values_catalog: dict | None = None,
    telco_retriever=None,
    include_all_threshold: int = 10,
) -> "Dataset":
    """
    Load enriched train_dataset.jsonl for GRPOTrainer.

    NOTE: Reward token injection happens at GENERATION time (per rollout),
    NOT at dataset creation time. The prompts stored here are reward-token-free.
    The GRPOTrainer subclass (or custom loop) must inject the reward token
    for each rollout in the group.
    """
    if Dataset is None or jsonlines is None:
        raise ImportError(
            "Required packages 'datasets' and/or 'jsonlines' are not installed."
        )

    raw_samples: list[dict] = []
    with jsonlines.open(jsonl_path) as reader:
        for obj in reader:
            raw_samples.append(obj)

    print(f"[base_trainer] Loaded {len(raw_samples)} samples from {jsonl_path}")

    val_retriever = None
    if telco_retriever is None and argument_values_catalog is not None:
        if ArgumentValueRetriever is not None:
            val_retriever = ArgumentValueRetriever(argument_values_catalog)

    formatted = []
    for sample in raw_samples:
        query = sample["query"]
        retrieved = sample.get("retrieved_functions", [])
        gt = sample.get("ground_truth", {})
        if not isinstance(gt, dict):
            gt = {}

        if telco_retriever is not None:
            result = telco_retriever.retrieve(
                query,
                function_library,
                precomputed_func_names=retrieved,
            )
            arg_vals = result.argument_values
        elif val_retriever is not None:
            arg_vals = val_retriever.retrieve_for_functions(
                query, retrieved, function_library
            )
        else:
            arg_vals = sample.get("retrieved_argument_values")

        formatted.append(
            format_sample_for_grpo(
                sample=sample,
                function_library=function_library,
                tokenizer=tokenizer,
                argument_values=arg_vals,
                include_all_threshold=include_all_threshold,
            )
        )

    import pandas as pd

    df = pd.DataFrame(formatted)
    dataset = Dataset.from_pandas(df)

    print(f"[base_trainer] GRPO dataset ready: {len(dataset)} samples")
    print(f"[base_trainer] Columns: {dataset.column_names}")

    # FIXED: ground_truth is now a JSON string (see format_sample_for_grpo).
    # Parse it back for display in the sanity check.
    if len(dataset) > 0:
        s = dataset[0]
        gt_raw = s.get("ground_truth", "{}")
        try:
            gt = json.loads(gt_raw) if isinstance(gt_raw, str) else gt_raw
        except json.JSONDecodeError:
            gt = {}
        wt = s.get("workflow_type", "single_call")
        print(f"\n{'=' * 70}")
        print("SAMPLE PROMPT (first 1500 chars):")
        print("=" * 70)
        print(s["prompt"][:1500])
        print("..." if len(s["prompt"]) > 1500 else "")
        calls = gt.get("calls", [])
        print(f"\nGROUND TRUTH (reward funcs see this, MODEL DOES NOT):")
        print(f"  workflow:  {wt}")
        print(f"  calls:     {len(calls)} steps")
        if calls:
            print(f"  calls:     {len(gt['calls'])} steps")
        print("=" * 70 + "\n")

    return dataset


def load_sft_dataset(
    jsonl_path: str,
    function_library: dict,
    tokenizer,
    argument_values_catalog: dict | None = None,
    telco_retriever=None,
) -> "Dataset":
    if Dataset is None or jsonlines is None:
        raise ImportError("datasets and/or jsonlines not installed.")

    raw_samples = []
    with jsonlines.open(jsonl_path) as reader:
        for obj in reader:
            raw_samples.append(obj)

    val_retriever = None
    if telco_retriever is None and argument_values_catalog is not None:
        if ArgumentValueRetriever is not None:
            val_retriever = ArgumentValueRetriever(argument_values_catalog)

    formatted = []
    for sample in raw_samples:
        query = sample["query"]
        retrieved = sample.get("retrieved_functions", [])

        if telco_retriever is not None:
            result = telco_retriever.retrieve(
                query, function_library, precomputed_func_names=retrieved
            )
            arg_vals = result.argument_values
        elif val_retriever is not None:
            arg_vals = val_retriever.retrieve_for_functions(
                query, retrieved, function_library
            )
        else:
            arg_vals = sample.get("retrieved_argument_values")

        formatted.append(
            format_sample_for_sft(
                sample, function_library, tokenizer, arg_vals
            )
        )

    dataset = Dataset.from_list(formatted)
    print(f"[base_trainer] SFT dataset ready: {len(dataset)} samples")
    return dataset

# ─────────────────────────────────────────────────────────────────────
# RC-GRPO–aware reward function factory for TRL GRPOTrainer
# BUG FIXED: imported from a nonexistent package `src.reward.base_reward`.
# This is a single-file script; format_reward / rc_grpo_reward_func /
# rc_grpo_format_func are already defined above in this same file.
# ─────────────────────────────────────────────────────────────────────

def build_trl_reward_functions(algorithm: str = "rc_grpo"):
    """
    Returns the list of reward functions compatible with TRL's GRPOTrainer
    for the selected algorithm.

    Each function signature: fn(completions: list[str], **kwargs) -> list[float]
    kwargs includes 'ground_truth' passed through from the dataset columns.
    """
    if algorithm == "rc_grpo":
        return [rc_grpo_reward_func, rc_grpo_format_func]
    # default / vanilla GRPO
    return [function_reward, argument_reward, format_reward]

# ─────────────────────────────────────────────────────────────────────
# Reward-token injection helper for TRL GRPOTrainer subclass
# ─────────────────────────────────────────────────────────────────────

def inject_reward_token_into_prompt(
    prompt: str,
    reward_token: str,
) -> str:
    """
    Inject [Reward Goal: <|high_reward|>] before the assistant generation
    prompt marker in a pre-built prompt string.

    For use when subclassing TRL's GRPOTrainer to add RC-GRPO support.
    """
    # Find the last <|im_start|>assistant marker and inject before it
    marker = "<|im_start|>assistant\n"
    idx = prompt.rfind(marker)
    if idx == -1:
        # fallback: append to end
        return prompt + f"\n[Reward Goal: {reward_token}]\n"
    inject_str = f"[Reward Goal: {reward_token}]\n"
    return prompt[:idx] + inject_str + prompt[idx:]

In [18]:
# ===================== metrics.py =====================
def compute_all_metrics(response: str, ground_truth: dict, sandbox, latency_ms: float,
                        cost_estimate: float, function_library: dict) -> dict[str, float]:
    workflow = ground_truth.get("workflow", "single_call")
    gold_calls = [
        {"function": c["function"], "arguments": c["arguments"]}
        for c in ground_truth.get("calls", [])
    ]
    agent_calls = extract_all_calls(response)
    schema_val = 1.0 if agent_calls else 0.0

    if gold_calls:
        func_hits = 0
        arg_scores = []
        for gc in gold_calls:
            best_func = 0.0
            best_arg = 0.0
            for ac in agent_calls:
                if ac.get("function") == gc["function"]:
                    best_func = 1.0
                    best_arg = max(best_arg, args_accuracy(ac, gc.get("arguments", {})))
            func_hits += best_func
            arg_scores.append(best_arg)
        func_sel_acc = func_hits / len(gold_calls)
        arg_acc = sum(arg_scores) / len(arg_scores)

        task_success = float(compute_action_coverage_reward(agent_calls, gold_calls))

        exec_success = 0.0
        if sandbox is not None:
            exec_success = 1.0 if sandbox.execute(response) else 0.0
        else:
            exec_success = float(len(agent_calls) > 0)
    else:
        func_sel_acc = 0.0
        arg_acc = 0.0
        task_success = 0.0
        exec_success = 0.0

    hallucinated = 0.0
    for ac in agent_calls:
        called_fn = ac.get("function", "")
        if called_fn and called_fn not in function_library:
            hallucinated = 1.0
            break

    abstention_acc = float("nan")
    if workflow == "abstention":
        produced_call = extract_call(response)
        abstention_acc = 1.0 if (produced_call is None or produced_call == "null") else 0.0

    return {
        "function_selection_accuracy": func_sel_acc,
        "argument_accuracy": arg_acc,
        "schema_validity": schema_val,
        "execution_success_rate": exec_success,
        "task_success_rate": task_success,
        "hallucinated_call_rate": hallucinated,
        "abstention_accuracy": abstention_acc,
        "latency_ms": latency_ms,
        "cost_per_query_usd": cost_estimate,
    }

def aggregate_metrics(results: list[dict[str, float]]) -> dict[str, float]:
    if not results:
        return {}
    keys = results[0].keys()
    agg = {}
    for k in keys:
        vals = [r[k] for r in results if not np.isnan(r[k])]
        if vals:
            agg[k] = float(np.mean(vals))
            agg[f"{k}__std"] = float(np.std(vals))
            agg[f"{k}__count"] = len(vals)
        else:
            agg[k] = float("nan")
    return agg

def estimate_cost(prompt: str, response: str, price_per_1k_tokens: float = 0.0002) -> float:
    total_chars = len(prompt) + len(response)
    tokens_est = total_chars / 1.3
    return (tokens_est / 1000) * price_per_1k_tokens

# ===================== benchmark.py =====================
# ===================== benchmark.py (FIXED) =====================
# BUG FIXED: paper Sec. 4.1 — "At inference, we condition on r =
# <|high_reward|> for optimal performance." The original evaluate_model
# built plain prompts with build_messages_for_grpo() and never injected
# the reward-goal token, so an RC-GRPO-trained model would be evaluated
# completely unconditioned (off-distribution relative to how it was
# trained to behave optimally).

def evaluate_model(
    model_path: str,
    test_dataset_path: str,
    function_library: dict,
    retriever: FunctionRetriever,
    sandbox: Sandbox,
    top_k: int = 5,
    max_new_tokens: int = 512,
    model_name_tag: str = "model",
    use_dataset_retrieval: bool = True,
    argument_values: dict | None = None,
    condition_on_high_reward: bool = True,
) -> dict:
    logger = get_logger(__name__)
    logger.info(f"[Benchmark] Loading model from {model_path}")
    model, tokenizer = load_model_from_path(model_path)
    patch_tokenizer_for_custom_roles(tokenizer)

    val_retriever = None
    if argument_values is not None:
        val_retriever = ArgumentValueRetriever(argument_values)

    test_samples = []
    with jsonlines.open(test_dataset_path) as reader:
        for obj in reader:
            test_samples.append(obj)

    logger.info(f"[Benchmark] {model_name_tag}: evaluating {len(test_samples)} samples")
    results = []
    for sample in tqdm(test_samples, desc=f"Eval [{model_name_tag}]"):
        query = sample["query"]
        gt = sample.get("ground_truth", {})
        if use_dataset_retrieval and sample.get("retrieved_functions"):
            retrieved = sample["retrieved_functions"]
        else:
            retrieved = retriever.retrieve(query, k=top_k)
        if val_retriever is not None:
            arg_vals = val_retriever.retrieve_for_functions(query, retrieved, function_library)
        else:
            arg_vals = sample.get("retrieved_argument_values")

        messages = build_messages_for_grpo(query, retrieved, function_library, arg_vals)

        # FIXED: condition on <|high_reward|> at inference (Sec. 4.1), only
        # meaningful/safe if the reward tokens are actually registered
        # (i.e. this checkpoint went through RCTP-FT / RC-GRPO).
        if condition_on_high_reward and HIGH_REWARD_TOKEN in tokenizer.all_special_tokens:
            messages = inject_reward_token_into_messages(messages, HIGH_REWARD_TOKEN)

        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        t0 = time.perf_counter()
        response = generate_response(model, tokenizer, prompt, max_new_tokens, temperature=0.0, do_sample=False)
        latency = (time.perf_counter() - t0) * 1000.0
        cost = estimate_cost(prompt, response)
        metrics = compute_all_metrics(response, gt, sandbox, latency, cost, function_library)
        metrics["sample_id"] = sample.get("id", "")
        results.append(metrics)

    agg = aggregate_metrics(results)
    logger.info(f"[Benchmark] {model_name_tag} aggregate: {agg}")
    return {"model": model_name_tag, "per_sample": results, "aggregate": agg}

In [19]:
# ===================== report_generator.py =====================
METRIC_DISPLAY_NAMES = {
    "function_selection_accuracy": "Func. Selection Acc.",
    "argument_accuracy": "Arg. Accuracy",
    "schema_validity": "Schema Validity",
    "execution_success_rate": "Exec. Success Rate",
    "task_success_rate": "Task Success Rate",
    "hallucinated_call_rate": "Hallucination Rate ↓",
    "abstention_accuracy": "Abstention Acc.",
    "latency_ms": "Latency (ms) ↓",
    "cost_per_query_usd": "Cost/Query (USD) ↓",
}

def generate_report(eval_results: list[dict], output_dir: str = "outputs/evaluation_reports") -> None:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    rows = []
    for result in eval_results:
        model = result["model"]
        agg = result["aggregate"]
        row = {"Model": model}
        for k, display in METRIC_DISPLAY_NAMES.items():
            row[display] = round(agg.get(k, float("nan")), 4)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("Model")
    print("\n" + "=" * 80)
    print("  EVALUATION REPORT — Telecom Tool-Calling with RL")
    print("=" * 80)
    print(tabulate(df, headers="keys", tablefmt="github", floatfmt=".4f"))
    csv_path = out / "metrics_summary.csv"
    df.to_csv(csv_path)
    print(f"[Report] CSV saved → {csv_path}")
    json_path = out / "full_results.json"
    with open(json_path, "w") as fh:
        json.dump(eval_results, fh, indent=2, default=str)
    print(f"[Report] JSON saved → {json_path}")
    # Optional plots
    try:
        _plot_bar_comparison(df, out)
        _plot_radar(df, out)
    except Exception as e:
        print(f"Plot generation failed: {e}")

def _plot_bar_comparison(df: pd.DataFrame, out: Path) -> None:
    core_metrics = ["Func. Selection Acc.", "Arg. Accuracy", "Schema Validity",
                    "Exec. Success Rate", "Task Success Rate"]
    plot_df = df[[c for c in core_metrics if c in df.columns]]
    fig, ax = plt.subplots(figsize=(12, 6))
    plot_df.T.plot(kind="bar", ax=ax, width=0.7, colormap="tab10")
    ax.set_title("Model Comparison – Core Metrics", fontsize=14, fontweight="bold")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.05)
    ax.legend(title="Model", loc="lower right")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    fig.savefig(out / "bar_comparison.png", dpi=150)
    plt.close(fig)

def _plot_radar(df: pd.DataFrame, out: Path) -> None:
    radar_metrics = ["Func. Selection Acc.", "Arg. Accuracy", "Schema Validity",
                     "Exec. Success Rate", "Task Success Rate", "Abstention Acc."]
    categories = [c for c in radar_metrics if c in df.columns]
    N = len(categories)
    if N < 3:
        return
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    fig, ax = plt.subplots(1, 1, figsize=(8, 8), subplot_kw={"polar": True})
    cmap = plt.get_cmap("tab10")
    for i, (model, row) in enumerate(df.iterrows()):
        vals = [row.get(c, 0.0) for c in categories]
        vals += vals[:1]
        ax.plot(angles, vals, "o-", linewidth=2, color=cmap(i), label=model)
        ax.fill(angles, vals, alpha=0.1, color=cmap(i))
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=10)
    ax.set_ylim(0, 1)
    ax.set_title("Algorithm Radar Comparison", size=14, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    fig.savefig(out / "radar_comparison.png", dpi=150)
    plt.close(fig)

In [20]:
# ===================== CONFIGURATION (FIXED) =====================
# BUG FIXED: ALGORITHM supported 'gvpo'/'autotool'/'awpo' branches below
# that referenced GVPOTrainer/AutoToolTrainer/AWPOTrainer/*_reward_func —
# none of which exist anywhere in this notebook (NameError at runtime).
# Those algorithms are NOT part of RC-GRPO (arXiv:2602.03025) and are out
# of scope for a "fix this to match the paper" pass, so they are removed
# rather than half-implemented. Only 'grpo' (vanilla baseline) and
# 'rc_grpo' (the paper's method) are supported now.
ALGORITHM = "rc_grpo"  # 'grpo' (vanilla baseline) or 'rc_grpo' (paper's method)
assert ALGORITHM in ("grpo", "rc_grpo"), (
    "Only 'grpo' (SFT+GRPO baseline) and 'rc_grpo' (RCTP-FT+RC-GRPO, the "
    "paper's full method) are implemented. gvpo/autotool/awpo were "
    "referenced in the original notebook but never defined."
)

DATA_DIR = Path("data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

FUNCTION_LIBRARY_PATH = DATA_DIR / "function_library.json"
ARGUMENT_VALUES_PATH = DATA_DIR / "argument_values.json"  # optional

TRAIN_CONFIG = {
    "model": {
        "name": "unsloth/Qwen3-4B-Base",
        "max_seq_length": 1024,
        "load_in_4bit": True,
        "fast_inference": True,
        "gpu_memory_utilization": 0.95,
    },
    "lora": {
        "r": 32,
        "lora_dropout": 0.0,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "bias": "none",
        "use_gradient_checkpointing": "unsloth",
    },
    "training": {
        "output_dir": f"outputs/{ALGORITHM}_model",
        # Table 10 (RC-GRPO RL stage hyperparameters)
        "learning_rate": 1e-6,
        "adam_beta1": 0.9,
        "adam_beta2": 0.99,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "lr_scheduler_type": "cosine",
        "optim": "adamw_8bit",
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 4,
        "num_generations": 5,         # Table 10: Group Size G = 5
        "max_steps": 50,             # reduce for quick test; increase for full run
        "save_steps": 50,
        "logging_steps": 1,
        "max_grad_norm": 1.0,
        "report_to": "wandb",
        "seed": 3407,
    },
    "data": {
        "train_path": "data/processed/train_dataset.jsonl",
        "test_path": "data/processed/test_dataset.jsonl",
        "max_prompt_length": 1024,
        "max_completion_length": 512,
        "include_all_threshold": 10,
    },
    "grpo": {
        "temperature": 1.0,
        "epsilon": 0.2,            # Table 10: symmetric Clip Ratio = 0.2
        "kl_coefficient": 0.1,     # Table 10: KL Coefficient beta = 0.1
        "loss_type": "grpo",
        "mask_truncated_completions": True,
    },
    # RCTP-FT (Stage 1) hyperparameters — Table 11
    "rctp_ft": {
        "learning_rate": 1e-5,
        "batch_size": 4,
        "gradient_accumulation_steps": 4,
        "num_epochs": 1,
        "output_dir": "outputs/rctp_ft_model",
        "failures_per_expert": 1,  # -> exact 1:1 success:failure ratio (Table 6)
    },
}

# Load function library and argument values (if available)
function_library = load_function_library(FUNCTION_LIBRARY_PATH)
print(f"Loaded {len(function_library)} functions")

argument_values = None
if ARGUMENT_VALUES_PATH.exists():
    with open(ARGUMENT_VALUES_PATH, "r") as f:
        argument_values = json.load(f)
    print(f"Loaded argument values catalog with {len(argument_values)} parameters")
else:
    print("No argument_values.json; will skip argument values in prompts.")


# ===================== Retriever Validation =====================
# if Path(TRAIN_CONFIG["data"]["test_path"]).exists():
#     with jsonlines.open(TRAIN_CONFIG["data"]["test_path"]) as reader:
#         test_samples = list(reader)
#     print(f"Loaded {len(test_samples)} test samples for retriever validation.")

#     retriever = FunctionRetriever(function_library, method="hybrid")
#     val_result = validate_retriever_consistency(
#         retriever=retriever,
#         test_samples=test_samples,
#         k=5,
#         n_repeats=10,
#         seed=42,
#     )
#     print("Retriever validation results:")
#     for k, v in val_result.items():
#         print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
# else:
#     print("Test dataset not found; skipping retriever validation.")


# ===================== Load model & register reward tokens =====================
model, tokenizer = load_model(TRAIN_CONFIG)
print("Model and tokenizer loaded.")

# Reward tokens are already registered + embeddings resized inside
# load_model() -> patch_tokenizer_for_custom_roles(). No separate
# RCGRPOTrainer.register_reward_tokens() call needed here (that method
# still exists and is idempotent if you want to call it again elsewhere).

print("Tokenizer special tokens:")
print(f"  bos_token: {tokenizer.bos_token}")
print(f"  eos_token: {tokenizer.eos_token}")
print(f"  pad_token: {tokenizer.pad_token}")
print(f"  additional_special_tokens: {tokenizer.additional_special_tokens}")

Loaded 31 functions
Loaded argument values catalog with 21 parameters
Unsloth: UnslothAlignPropTrainer is already patched.
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDDPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothIterativeSFTTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.
INFO 06-23 04:28:56 [vllm_utils.py:739] Unsloth: Patc

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 06-23 04:29:55 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 06-23 04:29:55 [config.py:1472] Using max model len 1024
WARNING 06-23 04:29:57 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 06-23 04:29:58 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=4096.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.0.mlp', 'model.layers.4.mlp', 'model.layers.3.self_attn', 'model.layers.0.self_attn', 'model.layers.6.mlp', 'model.layers.1.self_attn', 'model.layers.1.mlp', 'model.layers.2.mlp'], 'llm_int8_threshold': 6.0}
INFO 06-

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/qwen3-4b-base-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


[base_trainer] Added reward tokens: ['<|high_reward|>', '<|low_reward|>']
[base_trainer] Custom chat template registered (supports 'retriever' role).
[base_trainer] Tokenizer vocab size: 151672 (includes reward tokens: ['<|high_reward|>', '<|low_reward|>'])
Model and tokenizer loaded.
Tokenizer special tokens:
  bos_token: None
  eos_token: <|endoftext|>
  pad_token: <|PAD_TOKEN|>
  additional_special_tokens: ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>', '<|high_reward|>', '<|low_reward|>']


In [ ]:
# ===================== STAGE 1: RCTP-FT (only for rc_grpo) =====================
high_reward_probability = 0.5  # overwritten below from real data stats

if ALGORITHM == "rc_grpo":
    print("\n" + "=" * 70)
    print("STAGE 1: Reward-Conditioned Trajectory Policy (RCTP) Fine-tuning")
    print("=" * 70)

    rctp_cfg = TRAIN_CONFIG["rctp_ft"]

    rctp_trajectories = build_rctp_dataset_from_jsonl(
        jsonl_path=TRAIN_CONFIG["data"]["train_path"],
        function_library=function_library,
        argument_values_catalog=argument_values,
        failures_per_expert=rctp_cfg["failures_per_expert"],
        seed=TRAIN_CONFIG["training"]["seed"],
    )

    # Eq. 3: p should match the RCTP-FT dataset's empirical success rate
    # (Appendix B.1.1: "we set p = 0.5" because their set was exact 1:1).
    n_success = sum(1 for t in rctp_trajectories if t.reward == 1)
    n_total = len(rctp_trajectories)
    high_reward_probability = n_success / max(1, n_total)
    print(f"[Stage 1] p (high_reward_probability for Eq. 3) = {high_reward_probability:.4f} "
          f"({n_success}/{n_total} success)")

    rctp_dataset = RCTPDataset(rctp_trajectories, tokenizer, max_length=TRAIN_CONFIG["data"]["max_prompt_length"])
    rctp_loader = DataLoader(rctp_dataset, batch_size=rctp_cfg["batch_size"], shuffle=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=rctp_cfg["learning_rate"])
    num_training_steps = len(rctp_loader) * rctp_cfg["num_epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=max(1, len(rctp_loader)), num_training_steps=num_training_steps
    )

    model.train()
    accum_steps = rctp_cfg.get("gradient_accumulation_steps", 1)

    # Recompute total training steps for the scheduler
    total_batches = len(rctp_loader)
    total_optim_steps = (total_batches + accum_steps - 1) // accum_steps   # ceil
    num_training_steps = total_optim_steps * rctp_cfg["num_epochs"]

    # Recreate scheduler with corrected total steps
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, total_optim_steps),   # warmup over one epoch, adjust if needed
        num_training_steps=num_training_steps,
    )

    for epoch in range(rctp_cfg["num_epochs"]):
        total_loss = 0.0
        optimizer.zero_grad()   # start with zero gradients
        for batch_idx, batch in enumerate(tqdm(rctp_loader, desc=f"RCTP-FT epoch {epoch + 1}/{rctp_cfg['num_epochs']}")):
            input_ids = batch["input_ids"].to(model.device)
            attention_mask = batch["attention_mask"].to(model.device)
            labels = batch["labels"].to(model.device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            # Scale loss for accumulation
            loss = loss / accum_steps
            loss.backward()
            total_loss += loss.item() * accum_steps   # keep unscaled for reporting

            # Step every accum_steps batches (or at the end of the epoch)
            if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == total_batches:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        avg_loss = total_loss / total_batches
        print(f"[RCTP-FT] Epoch {epoch + 1}/{rctp_cfg['num_epochs']}  Loss: {avg_loss:.4f}")

    rctp_save_dir = Path(rctp_cfg["output_dir"])
    rctp_save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(rctp_save_dir)
    tokenizer.save_pretrained(rctp_save_dir)
    print(f"[Stage 1] RCTP-FT checkpoint saved -> {rctp_save_dir}")
    print("(This checkpoint becomes pi_ref for Stage 2, per Algorithm 1 line 9.)")

In [21]:
# ===================== STAGE 2: RC-GRPO (or vanilla GRPO baseline) =====================
print("\n" + "=" * 70)
print(f"STAGE 2: {'RC-GRPO' if ALGORITHM == 'rc_grpo' else 'Vanilla GRPO baseline'}")
print("=" * 70)

dataset = load_grpo_dataset(
    TRAIN_CONFIG["data"]["train_path"],
    function_library,
    tokenizer,
    argument_values_catalog=argument_values,
)

grpo_args = build_grpo_config(TRAIN_CONFIG, output_dir=TRAIN_CONFIG["training"]["output_dir"])

if ALGORITHM == "rc_grpo":
    trainer = RCGRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=build_trl_reward_functions("rc_grpo"),
        args=grpo_args,
        train_dataset=dataset,
        high_reward_probability=high_reward_probability,  # from Stage 1 stats, Eq. 3
    )
else:  # vanilla GRPO baseline (no reward conditioning) — for comparison only
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=build_trl_reward_functions("grpo"),
        args=grpo_args,
        train_dataset=dataset,
    )

print(f"Trainer initialized for {ALGORITHM}")

custom_tags = ["<reasoning>", "</reasoning>", "<tool_call>", "</tool_call>", HIGH_REWARD_TOKEN, LOW_REWARD_TOKEN]
existing = set(tokenizer.all_special_tokens)
conflicts = [tag for tag in custom_tags if tag in existing and tag not in REWARD_TOKENS]
if conflicts:
    print(f"WARNING: custom tags collide with existing special tokens: {conflicts}")
else:
    print("No conflicts detected with custom tags.")
print("Starting training...")
trainer.train()
print("Training completed.")


STAGE 2: RC-GRPO
[base_trainer] Loaded 3293 samples from data/processed/train_dataset.jsonl


ArrowInvalid: ('cannot mix list and non-list, non-null values', 'Conversion failed for column ground_truth with type object')

In [ ]:
# Save model and tokenizer
output_dir = grpo_args.output_dir
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

In [ ]:
# ===================== EVALUATION =====================
test_dataset_path = TRAIN_CONFIG["data"]["test_path"]
if Path(test_dataset_path).exists():
    retriever = FunctionRetriever(function_library, method="hybrid")
    sandbox = Sandbox(function_library)

    eval_result = evaluate_model(
        model_path=output_dir,
        test_dataset_path=test_dataset_path,
        function_library=function_library,
        retriever=retriever,
        sandbox=sandbox,
        top_k=5,
        max_new_tokens=512,
        model_name_tag=ALGORITHM,
        argument_values=argument_values,
    )
    print("Evaluation result:", eval_result["aggregate"])
    generate_report([eval_result])
else:
    print("Test dataset not found; skipping evaluation.")

In [ ]:
import torch
import gc
# Delete the model and tokenizer variables
del model
del tokenizer
# Trigger Python's garbage collection
gc.collect()
# Empty the PyTorch GPU cache
torch.cuda.empty_cache()

In [ ]:
!mkdir wheels
# Freeze everything EXCEPT google-colab
!uv pip freeze | grep -v "google-colab" > requirements.txt
!pip download -r requirements.txt -d wheels
!ls wheels/
!zip -r wheels.zip wheels/